# CC Media Studio — 从零自动准备

选择 **GPU T4 ×2** 并开启 Internet，然后运行所有单元格。 / Select **GPU T4 ×2**, enable Internet, and run all cells.

不需要已有 Dataset、模型文件或 Hugging Face Token。填提示词并点击按钮，首次自动下载公开权重，显示真实下载进度；之后复用已加载的模型。

如需跨会话缓存，展开可选设置，填写自己的 Kaggle 用户名与 API Key。首次自动创建私有 `用户名/qwen-image-studio-assets`，后续优先读取。留空仍可生图，权重仅保存在当前会话临时磁盘。不要保存带凭据的单元格输出。

In [ ]:
%pip install -q bitsandbytes==0.50.2 gguf requests ipywidgets 'diffusers>=0.36,<0.43' 'transformers>=4.51,<6' 'accelerate>=1.2'

In [ ]:
import base64, contextlib, html, io, json, time
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

"""Client-side model catalog. Importing it never loads GPU libraries."""
DEFAULT_MODEL = 'qwen-image-2.1'
MODELS = {
    DEFAULT_MODEL: {'id':DEFAULT_MODEL,'name':'Qwen-Image 2.1 · Quality','default_steps':20,'fixed_steps':None,
                    'cache_slug':'qwen-image-studio-assets','family':'qwen21'},
    'z-image-turbo': {'id':'z-image-turbo','name':'Z-Image Turbo · Efficient','default_steps':9,'fixed_steps':None,
                      'cache_slug':'cc-media-z-image-turbo','family':'zimage'},
    'sdxl-lightning': {'id':'sdxl-lightning','name':'SDXL-Lightning · 4-step Preview','default_steps':4,'fixed_steps':4,
                       'cache_slug':'cc-media-sdxl-lightning','family':'sdxl'},
}
MODEL_SOURCES = {
    'z-image-turbo': [('Tongyi-MAI/Z-Image-Turbo','f332072aa78be7aecdf3ee76d5c247082da564a6')],
    'sdxl-lightning': [('stabilityai/stable-diffusion-xl-base-1.0','462165984030d82259a11f4367a4eed129e94a7b'),
                       ('ByteDance/SDXL-Lightning','c9a24f48e1c025556787b0c58dd67a091ece2e44')],
}


def model_profile(model_id=None):
    model_id = DEFAULT_MODEL if model_id is None else model_id
    if not isinstance(model_id,str) or model_id not in MODELS:
        raise ValueError('未知模型 / Unknown model.')
    return dict(MODELS[model_id])


def model_steps(model_id, steps=None):
    profile=model_profile(model_id)
    steps=profile['default_steps'] if steps is None else steps
    if type(steps) is not int or not 1<=steps<=50:raise ValueError('steps 需要为 1–50 的整数。')
    if profile['fixed_steps'] is not None and steps!=profile['fixed_steps']:
        raise ValueError('SDXL-Lightning 固定使用 4 步 / requires 4 steps.')
    return steps


def model_catalog():return [dict(value) for value in MODELS.values()]

_BOOTSTRAPS = {'qwen-image-2.1': '\n_CC_REQUESTED_MODEL = \'qwen-image-2.1\'\n_CC_PREVIOUS_MODEL = globals().get(\'_CC_MODEL_ID\',\'qwen-image-2.1\')\nif _CC_PREVIOUS_MODEL != _CC_REQUESTED_MODEL:\n    if (globals().get(\'pipe\') is not None or globals().get(\'encoder\') is not None) and not (globals().get(\'_COMFY_QWEN_STATE\') or globals().get(\'_CC_MANAGED_MODEL\')):\n        raise RuntimeError(\'Existing unmanaged model; select the managed notebook kernel.\')\n    # Do not retain aliases to an old pipeline or its GPU modules during collection.\n    for _cc_key in (\'pipe\',\'encoder\',\'transformer\',\'vae\',\'processor\',\'scheduler\',\'generate_image\',\'_COMFY_QWEN_STATE\',\'_CC_NEGATIVE_CACHE\',\'_CC_PROMPT_CACHE\',\'_value\',\'_module\',\'_decoded\'):\n        globals().pop(_cc_key,None)\n    import gc\n    gc.collect()\n    try:\n        import torch\n        torch.cuda.empty_cache()\n    except ImportError:pass\n_CC_MODEL_ID = _CC_REQUESTED_MODEL\n_CC_MANAGED_MODEL = True\n\nimport json, os, uuid\nexec(compile(\'"""Download public HF weights with resumable transfers and real byte progress."""\\nimport hashlib\\nimport json\\nimport os\\nimport shutil\\nimport time\\nfrom pathlib import Path, PurePosixPath\\nfrom urllib.parse import quote\\nimport requests\\n\\nBASE_REPO = \\\'Qwen/Qwen-Image-2.1\\\'\\nGGUF_REPO = \\\'azukivc/Qwen-Image-2.1-GGUF\\\'\\nBASE_REVISION = \\\'d26bb61231c349cf6b7896fa83353113880e1ba3\\\'\\nGGUF_REVISION = \\\'4e43fd824763d3d24233630c343462592f68ca49\\\'\\n\\n\\ndef emit_progress(phase, message, done=None, total=None, filename=None):\\n    event = {\\\'phase\\\': phase, \\\'message\\\': message}\\n    if done is not None: event[\\\'downloaded_bytes\\\'] = done\\n    if total: event.update(total_bytes=total, percent=round(min(100, 100*done/total), 1))\\n    if filename: event[\\\'file\\\'] = filename\\n    callback = globals().get(\\\'_QWEN_PROGRESS_CALLBACK\\\')\\n    if callback: callback(event)\\n    print(\\\'COMFY_PROGRESS:\\\' + json.dumps(event, ensure_ascii=False), flush=True)\\n\\n\\ndef repository_manifest(repo, session=None, revision=None):\\n    http = session or requests.Session()\\n    endpoint = \\\'https://huggingface.co/api/models/\\\' + repo\\n    if revision: endpoint += \\\'/revision/\\\' + revision\\n    response = http.get(endpoint, params={\\\'blobs\\\':\\\'true\\\'}, timeout=(20,60))\\n    response.raise_for_status()\\n    data = response.json()\\n    if data.get(\\\'private\\\') or data.get(\\\'gated\\\'):\\n        raise RuntimeError(\\\'默认权重必须能够公开下载，当前来源需要额外授权。\\\')\\n    return data\\n\\n\\ndef safe_relative(name):\\n    path = PurePosixPath(name)\\n    if path.is_absolute() or \\\'..\\\' in path.parts or \\\'\\\\\\\\\\\' in name:\\n        raise ValueError(\\\'Invalid model file path\\\')\\n    return path\\n\\n\\ndef download_file(http, url, destination, expected_size, expected_sha=None, progress=None, verify_progress=None):\\n    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)\\n    partial = destination.with_name(destination.name + \\\'.partial\\\')\\n    receipt = destination.with_name(destination.name + \\\'.verified.json\\\')\\n    if destination.is_file() and destination.stat().st_size == expected_size:\\n        try:\\n            stamp = json.loads(receipt.read_text())\\n            if stamp == {\\\'size\\\':expected_size, \\\'sha256\\\':expected_sha}:\\n                if progress: progress(expected_size)\\n                return\\n        except (OSError,ValueError): pass\\n        # Existing files without a verification receipt must still be checked.\\n        partial.unlink(missing_ok=True); destination.replace(partial)\\n    for attempt in range(3):\\n        try:\\n            offset = partial.stat().st_size if partial.exists() else 0\\n            if expected_size and offset > expected_size:\\n                partial.unlink(); offset = 0\\n            if offset != expected_size or not partial.exists():\\n                headers = {\\\'Range\\\':f\\\'bytes={offset}-\\\'} if offset else {}\\n                with http.get(url, headers=headers, stream=True, timeout=(20,60)) as response:\\n                    response.raise_for_status()\\n                    if offset and response.status_code == 206:\\n                        if not response.headers.get(\\\'Content-Range\\\',\\\'\\\').startswith(f\\\'bytes {offset}-\\\'):\\n                            raise ValueError(\\\'Invalid download resume range\\\')\\n                    else:\\n                        offset = 0\\n                    with partial.open(\\\'ab\\\' if offset else \\\'wb\\\') as output:\\n                        for chunk in response.iter_content(chunk_size=4*1024*1024):\\n                            if chunk:\\n                                output.write(chunk); offset += len(chunk)\\n                                if progress: progress(offset)\\n            if expected_size and partial.stat().st_size != expected_size:\\n                raise OSError(\\\'Incomplete model download\\\')\\n            if expected_sha:\\n                digest = hashlib.sha256()\\n                with partial.open(\\\'rb\\\') as content:\\n                    verified = 0\\n                    for chunk in iter(lambda:content.read(16*1024*1024), b\\\'\\\'):\\n                        digest.update(chunk); verified += len(chunk)\\n                        if verify_progress: verify_progress(verified)\\n                if digest.hexdigest() != expected_sha:\\n                    partial.unlink(); raise ValueError(\\\'Model checksum mismatch\\\')\\n            partial.replace(destination)\\n            receipt.write_text(json.dumps({\\\'size\\\':expected_size,\\\'sha256\\\':expected_sha}))\\n            if progress: progress(destination.stat().st_size)\\n            return\\n        except (requests.RequestException, OSError, ValueError):\\n            if attempt == 2: raise\\n            time.sleep(2*(attempt+1))\\n\\n\\ndef prepare_hf_assets(folder, http=None):\\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\\n    http = http or requests.Session()\\n    emit_progress(\\\'manifest\\\', \\\'正在从 Hugging Face 获取模型文件清单。\\\')\\n    base = repository_manifest(BASE_REPO, http, BASE_REVISION)\\n    quant = repository_manifest(GGUF_REPO, http, GGUF_REVISION)\\n    records = []\\n    for item in base[\\\'siblings\\\']:\\n        name = item[\\\'rfilename\\\']\\n        if name in {\\\'model_index.json\\\',\\\'LICENSE\\\',\\\'transformer/config.json\\\'} or any(name.startswith(p+\\\'/\\\') for p in [\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'text_encoder\\\']):\\n            if name.endswith(\\\'.json\\\') or name.endswith(\\\'.safetensors\\\') or name.endswith(\\\'.txt\\\') or name.endswith(\\\'.model\\\') or name.endswith(\\\'.jinja\\\') or name.endswith(\\\'.tiktoken\\\') or name == \\\'LICENSE\\\':\\n                records.append((BASE_REPO,base[\\\'sha\\\'],item,name))\\n    candidates = [f for f in quant[\\\'siblings\\\'] if f[\\\'rfilename\\\'].lower().endswith(\\\'q4_k_m.gguf\\\')]\\n    if len(candidates) != 1: raise RuntimeError(\\\'兼容的 Q4_K_M 权重清单不明确。\\\')\\n    records.append((GGUF_REPO,quant[\\\'sha\\\'],candidates[0],\\\'transformer-q4.gguf\\\'))\\n    if not any(name.startswith(\\\'text_encoder/\\\') and name.endswith(\\\'.safetensors\\\') for _,_,_,name in records):\\n        raise RuntimeError(\\\'模型清单缺少文本编码器权重。\\\')\\n    total = sum(int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0) for _,_,item,_ in records)\\n    cached=True\\n    for _,_,item,name in records:\\n        safe_relative(name)\\n        size=int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        target=folder/name\\n        try:\\n            receipt=json.loads(target.with_name(target.name+\\\'.verified.json\\\').read_text())\\n            if not target.is_file() or target.stat().st_size!=size or receipt!={\\\'size\\\':size,\\\'sha256\\\':item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')}:\\n                cached=False;break\\n        except (OSError,ValueError):cached=False;break\\n    if cached:\\n        emit_progress(\\\'cached\\\',\\\'已找到完整模型缓存，无需重新下载。\\\',total,total or None)\\n        return folder\\n    done = 0; last = [0.0]\\n    for repo,revision,item,name in records:\\n        safe_relative(name)\\n        size = int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        sha = item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')\\n        start = done\\n        def progress(value, name=name, start=start):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'download\\\',\\\'首次准备：正在下载模型权重。\\\',start+value,total or None,name)\\n                last[0]=now\\n        url=\\\'https://huggingface.co/\\\'+repo+\\\'/resolve/\\\'+revision+\\\'/\\\'+quote(item[\\\'rfilename\\\'],safe=\\\'/\\\')\\n        def verify_progress(value, name=name, size=size):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'verify\\\',\\\'正在校验下载的权重。\\\',value,size,name);last[0]=now\\n        download_file(http,url,folder/name,size,sha,progress,verify_progress)\\n        done += size\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps({\\\'base_repo\\\':BASE_REPO,\\\'base_revision\\\':base[\\\'sha\\\'],\\\'gguf_repo\\\':GGUF_REPO,\\\'gguf_revision\\\':quant[\\\'sha\\\'],\\\'download_bytes\\\':total},indent=2))\\n    emit_progress(\\\'download\\\',\\\'模型权重下载完成。\\\',done,total or None)\\n    return folder\\n\\n\\ndef copy_assets(source, target):\\n    source,target=Path(source),Path(target)\\n    files=[p for p in source.rglob(\\\'*\\\') if p.is_file()]\\n    total=sum(p.stat().st_size for p in files); done=0; last=0\\n    for path in files:\\n        destination=target/path.relative_to(source); destination.parent.mkdir(parents=True,exist_ok=True)\\n        with path.open(\\\'rb\\\') as src, destination.open(\\\'wb\\\') as dst:\\n            for chunk in iter(lambda:src.read(8*1024*1024),b\\\'\\\'):\\n                dst.write(chunk);done+=len(chunk)\\n                now=time.monotonic()\\n                if now-last>=.8:\\n                    emit_progress(\\\'copy\\\',\\\'正在准备已缓存的模型文件。\\\',done,total,path.name);last=now\\n    emit_progress(\\\'copy\\\',\\\'模型文件准备完成。\\\',done,total)\\n\\n"""Private, create-only Kaggle cache of prepared INT8/Q4 model assets.\\n\\nExecuted in the user\\\'s Kaggle kernel. Credentials are passed at connection time,\\nused only for Kaggle operations, and removed by the caller\\\'s finally block.\\n"""\\nimport contextlib\\nimport tempfile\\nimport threading\\nfrom concurrent.futures import ThreadPoolExecutor\\n\\nCACHE_VERSION = 1\\n\\n\\ndef cache_file_hashes(folder):\\n    hashes = {}\\n    for file in Path(folder).rglob(\\\'*\\\'):\\n        if file.is_file():\\n            with file.open(\\\'rb\\\') as content:\\n                hashes[file.relative_to(folder).as_posix()] = hashlib.file_digest(content,\\\'sha256\\\').hexdigest()\\n    return hashes\\n\\n\\n@contextlib.contextmanager\\ndef cache_credentials(handle, token):\\n    owner = handle.split(\\\'/\\\')[0]\\n    values = {\\\'KAGGLE_USERNAME\\\': owner,\\\'KAGGLE_API_TOKEN\\\':None,\\\'KAGGLE_KEY\\\':None}\\n    if token.startswith(\\\'KGAT_\\\'): values[\\\'KAGGLE_API_TOKEN\\\'] = token\\n    else: values[\\\'KAGGLE_KEY\\\'] = token\\n    old = {name: os.environ.get(name) for name in values}\\n    try:\\n        for name,value in values.items():\\n            if value is None: os.environ.pop(name,None)\\n            else: os.environ[name]=value\\n        yield\\n    finally:\\n        for name, value in old.items():\\n            if value is None: os.environ.pop(name, None)\\n            else: os.environ[name] = value\\n\\n\\ndef cache_metadata(handle, token):\\n    owner, slug = handle.split(\\\'/\\\')\\n    with requests.Session() as http:\\n        if token.startswith(\\\'KGAT_\\\'): http.headers[\\\'Authorization\\\'] = \\\'Bearer \\\' + token\\n        else: http.auth = (owner, token)\\n        response = http.get(\\\'https://www.kaggle.com/api/v1/datasets/view/\\\'+owner+\\\'/\\\'+slug, timeout=(20,60))\\n        if response.status_code in (403,404):\\n            # Kaggle hides missing private slugs behind 403. Verify absence in\\n            # the authenticated owner listing; a permission failure stays fatal.\\n            for page in range(1,101):\\n                listing = http.get(\\\'https://www.kaggle.com/api/v1/datasets/list\\\',\\n                    params={\\\'user\\\':owner,\\\'search\\\':slug,\\\'page\\\':page}, timeout=(20,60))\\n                listing.raise_for_status()\\n                records = listing.json()\\n                if not isinstance(records,list): raise RuntimeError(\\\'Invalid dataset listing\\\')\\n                if any(item.get(\\\'ref\\\') == handle for item in records):\\n                    raise RuntimeError(\\\'Existing cache is inaccessible\\\')\\n                if len(records)<20: return None\\n            raise RuntimeError(\\\'Cannot confirm cache absence\\\')\\n        response.raise_for_status()\\n        data = response.json()\\n        if data.get(\\\'isPrivate\\\') is not True:\\n            raise RuntimeError(\\\'模型缓存必须为私有 Dataset；没有修改现有 Dataset。\\\')\\n        return data\\n\\n\\ndef validate_cache_manifest(manifest):\\n    generic=manifest.get(\\\'studio_model_id\\\') is not None\\n    if generic:\\n        if (manifest.get(\\\'studio_cache_version\\\')!=2 or manifest[\\\'studio_model_id\\\']!=globals().get(\\\'_CC_MODEL_ID\\\')\\n            or manifest.get(\\\'model_sources\\\')!=globals().get(\\\'_CC_MODEL_SOURCES\\\')):\\n            raise RuntimeError(\\\'所选模型缓存版本不兼容。\\\')\\n    elif (manifest.get(\\\'studio_cache_version\\\') != CACHE_VERSION\\n        or manifest.get(\\\'base_revision\\\') != BASE_REVISION\\n        or manifest.get(\\\'gguf_revision\\\') != GGUF_REVISION):\\n        raise RuntimeError(\\\'同名 Dataset 不是兼容的本程序缓存，没有覆盖它。\\\')\\n    files = manifest.get(\\\'cache_files\\\', {})\\n    if not files or (not generic and \\\'transformer-q4.gguf\\\' not in files) or (generic and not any(name.endswith(\\\'.safetensors\\\') for name in files)):\\n        raise RuntimeError(\\\'模型缓存文件清单不完整。\\\')\\n    for name, size in files.items():\\n        safe_relative(name)\\n        if not isinstance(size,int) or size<0: raise RuntimeError(\\\'Invalid cached file size\\\')\\n    return manifest\\n\\n\\ndef validate_cache(folder):\\n    folder = Path(folder)\\n    manifest = validate_cache_manifest(json.loads((folder/\\\'asset-sources.json\\\').read_text()))\\n    for name, size in manifest[\\\'cache_files\\\'].items():\\n        file = folder/name\\n        if not file.is_file() or file.stat().st_size != size:\\n            raise RuntimeError(\\\'模型缓存文件不完整。\\\')\\n    return manifest\\n\\n\\ndef cache_http(handle, token):\\n    http = requests.Session()\\n    if token.startswith(\\\'KGAT_\\\'): http.headers[\\\'Authorization\\\'] = \\\'Bearer \\\'+token\\n    else: http.auth = (handle.split(\\\'/\\\')[0],token)\\n    return http\\n\\n\\ndef cache_file_url(handle, version, name):\\n    safe_relative(name)\\n    return (\\\'https://www.kaggle.com/api/v1/datasets/download/\\\'+handle+\\\'/\\\'+quote(name,safe=\\\'\\\')+\\n        \\\'?raw=true&dataset_version_number=\\\'+str(int(version)))\\n\\n\\ndef read_cache_manifest(http, handle, version):\\n    response = http.get(cache_file_url(handle,version,\\\'asset-sources.json\\\'),timeout=(20,60))\\n    response.raise_for_status()\\n    return validate_cache_manifest(response.json())\\n\\n\\ndef download_cache_assets(folder, handle, token, metadata):\\n    folder = Path(folder); folder.mkdir(parents=True,exist_ok=True)\\n    version = metadata[\\\'currentVersionNumber\\\']\\n    with cache_http(handle,token) as http:\\n        manifest = read_cache_manifest(http,handle,version)\\n    total = sum(manifest[\\\'cache_files\\\'].values()); received={}; last=[0.0]\\n    progress_lock=threading.Lock()\\n    def transfer(item):\\n        name,size=item\\n        def progress(value):\\n            with progress_lock:\\n                received[name]=value\\n                now=time.monotonic()\\n                if now-last[0]>=.8 or value==size:\\n                    emit_progress(\\\'cache-download\\\',\\\'正在拉取私有 Dataset 模型缓存。\\\',sum(received.values()),total,name)\\n                    last[0]=now\\n        with cache_http(handle,token) as http:\\n            download_file(http,cache_file_url(handle,version,name),folder/name,size,\\n                manifest.get(\\\'cache_sha256\\\',{}).get(name),progress)\\n    # Independent resumable streams avoid serializing all multi-GB shards.\\n    files=sorted(manifest[\\\'cache_files\\\'].items(),key=lambda item:item[1],reverse=True)\\n    with ThreadPoolExecutor(max_workers=4) as pool:\\n        list(pool.map(transfer,files))\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps(manifest,indent=2))\\n    validate_cache(folder)\\n    return folder\\n\\n\\ndef prepare_auto_assets(folder):\\n    handle, token = globals().get(\\\'_QWEN_CACHE_HANDLE\\\'), globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'disabled\\\'\\n    globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\'] = handle\\n    if handle and token:\\n        try:\\n            emit_progress(\\\'cache-check\\\', \\\'正在检查你的私有模型缓存。\\\')\\n            metadata = cache_metadata(handle, token)\\n            if metadata:\\n                emit_progress(\\\'cache-download\\\', \\\'正在拉取私有 Dataset 模型缓存。\\\')\\n                download_cache_assets(folder, handle, token, metadata)\\n                globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n                globals()[\\\'_QWEN_RESOLVED_SOURCE\\\'] = \\\'kaggle-cache\\\'\\n                return Path(folder), False\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'missing\\\'\\n        except Exception:\\n            # A metadata/auth/compatibility error must never permit overwriting\\n            # an existing dataset. Generation can still use the public weights.\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'unavailable\\\'\\n            emit_progress(\\\'cache-warning\\\', \\\'私有缓存暂不可用，本次从 Hugging Face 准备；没有修改已有 Dataset。\\\')\\n    globals()[\\\'_QWEN_RESOLVED_SOURCE\\\'] = \\\'huggingface\\\'\\n    return prepare_hf_assets(folder), True\\n\\n\\ndef save_private_cache(folder, encoder):\\n    handle, token = globals().get(\\\'_QWEN_CACHE_HANDLE\\\'), globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    status = globals().get(\\\'_QWEN_CACHE_STATUS\\\', \\\'missing\\\') if globals().get(\\\'_QWEN_CACHE_STATUS_HANDLE\\\') == handle else \\\'missing\\\'\\n    globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\'] = handle\\n    if not handle or not token or status in (\\\'saved\\\',\\\'unavailable\\\'): return\\n    try:\\n        emit_progress(\\\'cache-check\\\', \\\'正在确认私有缓存状态。\\\')\\n        metadata = cache_metadata(handle, token)\\n        if metadata:\\n            with cache_http(handle,token) as http:\\n                read_cache_manifest(http,handle,metadata[\\\'currentVersionNumber\\\'])\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n            return\\n        folder = Path(folder)\\n        emit_progress(\\\'cache-prepare\\\', \\\'正在保存 INT8 编码器和 Q4 权重，首次需要额外等待。\\\')\\n        with tempfile.TemporaryDirectory(prefix=\\\'qwen-private-cache-\\\', dir=\\\'/kaggle/temp\\\') as temporary:\\n            bundle = Path(temporary)\\n            generic=globals().get(\\\'_CC_MODEL_ID\\\',\\\'qwen-image-2.1\\\')!=\\\'qwen-image-2.1\\\'\\n            if generic:\\n                def link_or_copy(src,dst):\\n                    try:os.link(src,dst)\\n                    except OSError:shutil.copy2(src,dst)\\n                shutil.copytree(folder,bundle,dirs_exist_ok=True,copy_function=link_or_copy,\\n                    ignore=shutil.ignore_patterns(\\\'*.partial\\\',\\\'*.verified.json\\\',\\\'asset-sources.json\\\'))\\n            for name in (() if generic else (\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'transformer\\\')):\\n                shutil.copytree(folder/name, bundle/name, ignore=shutil.ignore_patterns(\\\'*.partial\\\',\\\'*.verified.json\\\'))\\n            for name in (() if generic else (\\\'transformer-q4.gguf\\\',\\\'model_index.json\\\',\\\'LICENSE\\\')):\\n                if (folder/name).exists(): shutil.copy2(folder/name, bundle/name)\\n            if not generic:encoder.save_pretrained(bundle/\\\'text_encoder\\\', safe_serialization=True, max_shard_size=\\\'4GB\\\')\\n            manifest = json.loads((folder/\\\'asset-sources.json\\\').read_text())\\n            manifest.update(studio_cache_version=2 if generic else CACHE_VERSION,\\n                cache_files={p.relative_to(bundle).as_posix():p.stat().st_size for p in bundle.rglob(\\\'*\\\') if p.is_file()},\\n                cache_sha256=cache_file_hashes(bundle))\\n            (bundle/\\\'asset-sources.json\\\').write_text(json.dumps(manifest, indent=2))\\n            validate_cache(bundle)\\n            emit_progress(\\\'cache-upload\\\', \\\'正在上传私有 Dataset；Kaggle 将显示文件上传进度。\\\')\\n            # Use KaggleHub\\\'s create-only primitive: its public upload helper\\n            # creates a new version on conflict, which could overwrite user data.\\n            from kagglehub import datasets\\n            api = datasets.dataset_upload.__globals__\\n            create = datasets.create_dataset_or_version.__globals__[\\\'_create_dataset\\\']\\n            with cache_credentials(handle, token):\\n                uploads = api[\\\'upload_files_and_directories\\\'](str(bundle), item_type=api[\\\'ApiBlobType\\\'].DATASET,\\n                    ignore_patterns=api[\\\'normalize_patterns\\\'](default=api[\\\'DEFAULT_IGNORE_PATTERNS\\\'], additional=None))\\n                create(api[\\\'parse_dataset_handle\\\'](handle), uploads)\\n            # The create primitive explicitly sends is_private=True. Confirm\\n            # privacy again from the server before reporting success.\\n            emit_progress(\\\'cache-processing\\\', \\\'文件已上传，Kaggle 正在处理私有 Dataset。\\\')\\n            deadline = time.monotonic()+180\\n            while not cache_metadata(handle, token):\\n                if time.monotonic() >= deadline: raise RuntimeError(\\\'Dataset metadata not ready\\\')\\n                time.sleep(5)\\n        globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n        emit_progress(\\\'cache-saved\\\', \\\'私有模型缓存已保存，下次新会话优先拉取，无需重新下载或量化。\\\')\\n    except Exception as error:\\n        globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'failed\\\'\\n        globals()[\\\'_QWEN_CACHE_FAILURE_TYPE\\\'] = type(error).__name__\\n        emit_progress(\\\'cache-warning\\\', \\\'私有缓存保存未完成；本次仍可正常生图，重新连接后会再次检查。\\\')\\n\', \'<qwen-private-cache>\', \'exec\'), globals())\n_expected_dataset = \'huggingface/qwen-image-21\'\n_existing_state = globals().get(\'_COMFY_QWEN_STATE\')\n_model_ready = (globals().get(\'pipe\') is not None and globals().get(\'encoder\') is not None\n                and callable(globals().get(\'generate_image\')))\nif _existing_state and _existing_state[\'dataset\'] != _expected_dataset:\n    if \'auto\' in (\'huggingface\',\'auto\') and _existing_state[\'dataset\'] == \'qwen-studio/huggingface\' and globals().get(\'_hf_mode\') is True:\n        _existing_state[\'dataset\'] = _expected_dataset\n    else:\n        raise RuntimeError(\'This notebook holds a different dataset. Use its existing dataset or restart the notebook explicitly.\')\nif _model_ready:\n    if not _existing_state:\n        if _expected_dataset not in str(globals().get(\'source\', \'\')):\n            raise RuntimeError(\'An existing model was found but its dataset cannot be verified. Select the correct notebook kernel; do not load a second model.\')\n        _existing_state = {\'dataset\': _expected_dataset, \'runtime_id\': uuid.uuid4().hex,\n                           \'load_count\': 1, \'generations\': 0, \'adopted\': True}\n    _existing_state[\'loaded_this_request\'] = False\nelse:\n    if globals().get(\'pipe\') is not None or globals().get(\'encoder\') is not None:\n        raise RuntimeError(\'A partially loaded model exists. Restart this notebook session before loading again.\')\n    if False:\n        import subprocess, sys\n        print(\'Installing image runtime dependencies once...\', flush=True)\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'bitsandbytes==0.50.2\', \'gguf\'], check=True, timeout=900)\n    exec(compile(\'"""Download public HF weights with resumable transfers and real byte progress."""\\nimport hashlib\\nimport json\\nimport os\\nimport shutil\\nimport time\\nfrom pathlib import Path, PurePosixPath\\nfrom urllib.parse import quote\\nimport requests\\n\\nBASE_REPO = \\\'Qwen/Qwen-Image-2.1\\\'\\nGGUF_REPO = \\\'azukivc/Qwen-Image-2.1-GGUF\\\'\\nBASE_REVISION = \\\'d26bb61231c349cf6b7896fa83353113880e1ba3\\\'\\nGGUF_REVISION = \\\'4e43fd824763d3d24233630c343462592f68ca49\\\'\\n\\n\\ndef emit_progress(phase, message, done=None, total=None, filename=None):\\n    event = {\\\'phase\\\': phase, \\\'message\\\': message}\\n    if done is not None: event[\\\'downloaded_bytes\\\'] = done\\n    if total: event.update(total_bytes=total, percent=round(min(100, 100*done/total), 1))\\n    if filename: event[\\\'file\\\'] = filename\\n    callback = globals().get(\\\'_QWEN_PROGRESS_CALLBACK\\\')\\n    if callback: callback(event)\\n    print(\\\'COMFY_PROGRESS:\\\' + json.dumps(event, ensure_ascii=False), flush=True)\\n\\n\\ndef repository_manifest(repo, session=None, revision=None):\\n    http = session or requests.Session()\\n    endpoint = \\\'https://huggingface.co/api/models/\\\' + repo\\n    if revision: endpoint += \\\'/revision/\\\' + revision\\n    response = http.get(endpoint, params={\\\'blobs\\\':\\\'true\\\'}, timeout=(20,60))\\n    response.raise_for_status()\\n    data = response.json()\\n    if data.get(\\\'private\\\') or data.get(\\\'gated\\\'):\\n        raise RuntimeError(\\\'默认权重必须能够公开下载，当前来源需要额外授权。\\\')\\n    return data\\n\\n\\ndef safe_relative(name):\\n    path = PurePosixPath(name)\\n    if path.is_absolute() or \\\'..\\\' in path.parts or \\\'\\\\\\\\\\\' in name:\\n        raise ValueError(\\\'Invalid model file path\\\')\\n    return path\\n\\n\\ndef download_file(http, url, destination, expected_size, expected_sha=None, progress=None, verify_progress=None):\\n    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)\\n    partial = destination.with_name(destination.name + \\\'.partial\\\')\\n    receipt = destination.with_name(destination.name + \\\'.verified.json\\\')\\n    if destination.is_file() and destination.stat().st_size == expected_size:\\n        try:\\n            stamp = json.loads(receipt.read_text())\\n            if stamp == {\\\'size\\\':expected_size, \\\'sha256\\\':expected_sha}:\\n                if progress: progress(expected_size)\\n                return\\n        except (OSError,ValueError): pass\\n        # Existing files without a verification receipt must still be checked.\\n        partial.unlink(missing_ok=True); destination.replace(partial)\\n    for attempt in range(3):\\n        try:\\n            offset = partial.stat().st_size if partial.exists() else 0\\n            if expected_size and offset > expected_size:\\n                partial.unlink(); offset = 0\\n            if offset != expected_size or not partial.exists():\\n                headers = {\\\'Range\\\':f\\\'bytes={offset}-\\\'} if offset else {}\\n                with http.get(url, headers=headers, stream=True, timeout=(20,60)) as response:\\n                    response.raise_for_status()\\n                    if offset and response.status_code == 206:\\n                        if not response.headers.get(\\\'Content-Range\\\',\\\'\\\').startswith(f\\\'bytes {offset}-\\\'):\\n                            raise ValueError(\\\'Invalid download resume range\\\')\\n                    else:\\n                        offset = 0\\n                    with partial.open(\\\'ab\\\' if offset else \\\'wb\\\') as output:\\n                        for chunk in response.iter_content(chunk_size=4*1024*1024):\\n                            if chunk:\\n                                output.write(chunk); offset += len(chunk)\\n                                if progress: progress(offset)\\n            if expected_size and partial.stat().st_size != expected_size:\\n                raise OSError(\\\'Incomplete model download\\\')\\n            if expected_sha:\\n                digest = hashlib.sha256()\\n                with partial.open(\\\'rb\\\') as content:\\n                    verified = 0\\n                    for chunk in iter(lambda:content.read(16*1024*1024), b\\\'\\\'):\\n                        digest.update(chunk); verified += len(chunk)\\n                        if verify_progress: verify_progress(verified)\\n                if digest.hexdigest() != expected_sha:\\n                    partial.unlink(); raise ValueError(\\\'Model checksum mismatch\\\')\\n            partial.replace(destination)\\n            receipt.write_text(json.dumps({\\\'size\\\':expected_size,\\\'sha256\\\':expected_sha}))\\n            if progress: progress(destination.stat().st_size)\\n            return\\n        except (requests.RequestException, OSError, ValueError):\\n            if attempt == 2: raise\\n            time.sleep(2*(attempt+1))\\n\\n\\ndef prepare_hf_assets(folder, http=None):\\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\\n    http = http or requests.Session()\\n    emit_progress(\\\'manifest\\\', \\\'正在从 Hugging Face 获取模型文件清单。\\\')\\n    base = repository_manifest(BASE_REPO, http, BASE_REVISION)\\n    quant = repository_manifest(GGUF_REPO, http, GGUF_REVISION)\\n    records = []\\n    for item in base[\\\'siblings\\\']:\\n        name = item[\\\'rfilename\\\']\\n        if name in {\\\'model_index.json\\\',\\\'LICENSE\\\',\\\'transformer/config.json\\\'} or any(name.startswith(p+\\\'/\\\') for p in [\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'text_encoder\\\']):\\n            if name.endswith(\\\'.json\\\') or name.endswith(\\\'.safetensors\\\') or name.endswith(\\\'.txt\\\') or name.endswith(\\\'.model\\\') or name.endswith(\\\'.jinja\\\') or name.endswith(\\\'.tiktoken\\\') or name == \\\'LICENSE\\\':\\n                records.append((BASE_REPO,base[\\\'sha\\\'],item,name))\\n    candidates = [f for f in quant[\\\'siblings\\\'] if f[\\\'rfilename\\\'].lower().endswith(\\\'q4_k_m.gguf\\\')]\\n    if len(candidates) != 1: raise RuntimeError(\\\'兼容的 Q4_K_M 权重清单不明确。\\\')\\n    records.append((GGUF_REPO,quant[\\\'sha\\\'],candidates[0],\\\'transformer-q4.gguf\\\'))\\n    if not any(name.startswith(\\\'text_encoder/\\\') and name.endswith(\\\'.safetensors\\\') for _,_,_,name in records):\\n        raise RuntimeError(\\\'模型清单缺少文本编码器权重。\\\')\\n    total = sum(int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0) for _,_,item,_ in records)\\n    cached=True\\n    for _,_,item,name in records:\\n        safe_relative(name)\\n        size=int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        target=folder/name\\n        try:\\n            receipt=json.loads(target.with_name(target.name+\\\'.verified.json\\\').read_text())\\n            if not target.is_file() or target.stat().st_size!=size or receipt!={\\\'size\\\':size,\\\'sha256\\\':item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')}:\\n                cached=False;break\\n        except (OSError,ValueError):cached=False;break\\n    if cached:\\n        emit_progress(\\\'cached\\\',\\\'已找到完整模型缓存，无需重新下载。\\\',total,total or None)\\n        return folder\\n    done = 0; last = [0.0]\\n    for repo,revision,item,name in records:\\n        safe_relative(name)\\n        size = int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        sha = item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')\\n        start = done\\n        def progress(value, name=name, start=start):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'download\\\',\\\'首次准备：正在下载模型权重。\\\',start+value,total or None,name)\\n                last[0]=now\\n        url=\\\'https://huggingface.co/\\\'+repo+\\\'/resolve/\\\'+revision+\\\'/\\\'+quote(item[\\\'rfilename\\\'],safe=\\\'/\\\')\\n        def verify_progress(value, name=name, size=size):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'verify\\\',\\\'正在校验下载的权重。\\\',value,size,name);last[0]=now\\n        download_file(http,url,folder/name,size,sha,progress,verify_progress)\\n        done += size\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps({\\\'base_repo\\\':BASE_REPO,\\\'base_revision\\\':base[\\\'sha\\\'],\\\'gguf_repo\\\':GGUF_REPO,\\\'gguf_revision\\\':quant[\\\'sha\\\'],\\\'download_bytes\\\':total},indent=2))\\n    emit_progress(\\\'download\\\',\\\'模型权重下载完成。\\\',done,total or None)\\n    return folder\\n\\n\\ndef copy_assets(source, target):\\n    source,target=Path(source),Path(target)\\n    files=[p for p in source.rglob(\\\'*\\\') if p.is_file()]\\n    total=sum(p.stat().st_size for p in files); done=0; last=0\\n    for path in files:\\n        destination=target/path.relative_to(source); destination.parent.mkdir(parents=True,exist_ok=True)\\n        with path.open(\\\'rb\\\') as src, destination.open(\\\'wb\\\') as dst:\\n            for chunk in iter(lambda:src.read(8*1024*1024),b\\\'\\\'):\\n                dst.write(chunk);done+=len(chunk)\\n                now=time.monotonic()\\n                if now-last>=.8:\\n                    emit_progress(\\\'copy\\\',\\\'正在准备已缓存的模型文件。\\\',done,total,path.name);last=now\\n    emit_progress(\\\'copy\\\',\\\'模型文件准备完成。\\\',done,total)\\n\\n_QWEN_ASSET_MODE = \\\'auto\\\'\\nimport json, shutil, time\\nfrom pathlib import Path\\nimport torch\\nimport kagglehub\\nfrom transformers import Qwen3VLForConditionalGeneration, Qwen3VLProcessor\\nfrom diffusers import (QwenImage21Pipeline, QwenImage21Transformer2DModel,\\n                       AutoencoderKLQwenImage21, FlowMatchEulerDiscreteScheduler,\\n                       GGUFQuantizationConfig)\\n\\nassert torch.cuda.device_count() == 2, \\\'Select GPU T4 x2 before running.\\\'\\nstarted = time.perf_counter()\\nASSET_DIR = Path(\\\'/kaggle/temp/qwen-image-assets\\\')\\n_hf_mode = globals().get(\\\'_QWEN_ASSET_MODE\\\') in (\\\'huggingface\\\',\\\'auto\\\')\\nif globals().get(\\\'_QWEN_ASSET_MODE\\\') == \\\'auto\\\':\\n    source, _hf_mode = prepare_auto_assets(ASSET_DIR)\\nelif _hf_mode:\\n    source = prepare_hf_assets(ASSET_DIR)\\nelse:\\n    emit_progress(\\\'dataset\\\', \\\'正在拉取 Kaggle 模型 Dataset。\\\')\\n    source = Path(kagglehub.dataset_download(\\\'huggingface/qwen-image-21\\\'))\\n    copy_assets(source, ASSET_DIR)\\ncopy_seconds = time.perf_counter()-started\\nprint(f\\\'Local assets ready: {copy_seconds:.2f} seconds\\\', flush=True)\\nprint(\\\'Loading INT8 text encoder on GPU 1...\\\', flush=True)\\nfrom transformers import BitsAndBytesConfig\\n_encoder_options = {\\\'quantization_config\\\': BitsAndBytesConfig(load_in_8bit=True)} if _hf_mode else {}\\nencoder = Qwen3VLForConditionalGeneration.from_pretrained(\\n    ASSET_DIR/\\\'text_encoder\\\', dtype=torch.float16, device_map={\\\'\\\': 1},\\n    attn_implementation=\\\'sdpa\\\', local_files_only=True, **_encoder_options)\\nprint(\\\'Loading Q4 transformer on GPU 0...\\\', flush=True)\\ntransformer = QwenImage21Transformer2DModel.from_single_file(\\n    str(ASSET_DIR/\\\'transformer-q4.gguf\\\'), config=str(ASSET_DIR), subfolder=\\\'transformer\\\',\\n    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float16),\\n    dtype=torch.float16, local_files_only=True).to(\\\'cuda:0\\\')\\n# Diffusers GGUFLinear handles packed matrices; custom RMSNorm vectors need\\n# explicit decoding when a pre-converted GGUF stores them as BF16 bytes.\\nfrom diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\\n_gguf_vector_fixes = 0\\nfor _name, _value in list(transformer.named_parameters()):\\n    if _value.ndim == 1 and hasattr(_value, \\\'quant_type\\\'):\\n        _parent, _, _field = _name.rpartition(\\\'.\\\')\\n        _module = transformer.get_submodule(_parent) if _parent else transformer\\n        _decoded = dequantize_gguf_tensor(_value).to(dtype=torch.float16)\\n        _logical_shape = getattr(_value, \\\'quant_shape\\\', None)\\n        if _logical_shape is not None:\\n            _decoded = _decoded.reshape(tuple(_logical_shape))\\n        setattr(_module, _field, torch.nn.Parameter(_decoded, requires_grad=False))\\n        _gguf_vector_fixes += 1\\nprint(\\\'GGUF vector compatibility:\\\', _gguf_vector_fixes, flush=True)\\nprint(\\\'Loading VAE and processor...\\\', flush=True)\\nvae = AutoencoderKLQwenImage21.from_pretrained(\\n    ASSET_DIR/\\\'vae\\\', dtype=torch.float16, local_files_only=True).to(\\\'cuda:0\\\')\\nprocessor = Qwen3VLProcessor.from_pretrained(ASSET_DIR/\\\'processor\\\', local_files_only=True)\\nscheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(\\n    ASSET_DIR/\\\'scheduler\\\', local_files_only=True)\\nindex = json.loads((ASSET_DIR/\\\'model_index.json\\\').read_text())\\npipe = QwenImage21Pipeline(scheduler=scheduler, vae=vae, text_encoder=encoder,\\n    processor=processor, transformer=transformer, sample_sigmas=index.get(\\\'sample_sigmas\\\'))\\nload_stats = {\\\'copy_seconds\\\': round(copy_seconds, 2),\\n              \\\'loading_seconds\\\': round(time.perf_counter()-started-copy_seconds, 2)}\\nprint(\\\'Pipeline ready:\\\', load_stats, flush=True)\\nfor i in range(2):\\n    print(f\\\'GPU {i}: {torch.cuda.memory_allocated(i)/2**30:.2f} GiB allocated\\\')\\n\\nimport json, time, torch\\n\\nPROMPT = "A small orange fox sitting beside a ceramic teacup, soft morning window light, detailed natural photography, clean background"\\nSEED = 42\\nWIDTH = HEIGHT = 512\\nSTEPS = 35\\n\\n@torch.inference_mode()\\ndef generate_image(prompt, seed=42, width=512, height=512, steps=35):\\n    for i in range(2):\\n        torch.cuda.reset_peak_memory_stats(i)\\n    started = time.perf_counter()\\n    pipe.register_modules(text_encoder=encoder)\\n    positive = pipe.encode_prompt(prompt, device=torch.device(\\\'cuda:1\\\'))\\n    negative = pipe.encode_prompt(\\\' \\\', device=torch.device(\\\'cuda:1\\\'))\\n    torch.cuda.synchronize(1)\\n    encoded = time.perf_counter()\\n    positive = tuple(t.to(\\\'cuda:0\\\') if t is not None else None for t in positive)\\n    negative = tuple(t.to(\\\'cuda:0\\\') if t is not None else None for t in negative)\\n    pipe.register_modules(text_encoder=None)\\n    _step_events=[]\\n    def _record_step(pipeline,index,timestep,callback_kwargs):\\n        event=torch.cuda.Event(enable_timing=True);event.record(torch.cuda.current_stream(0));_step_events.append(event)\\n        return callback_kwargs\\n    _before=torch.cuda.Event(enable_timing=True);_before.record(torch.cuda.current_stream(0))\\n    try:\\n        result = pipe(\\n            prompt_embeds=positive[0], prompt_embeds_mask=positive[1],\\n            negative_prompt_embeds=negative[0], negative_prompt_embeds_mask=negative[1],\\n            width=width, height=height, num_inference_steps=steps,\\n            sigmas=torch.linspace(1, 0, steps+1)[:-1].tolist(),\\n            true_cfg_scale=5.0, use_kv_cache=True,callback_on_step_end=_record_step,\\n            generator=torch.Generator(device=\\\'cuda:0\\\').manual_seed(seed)).images[0]\\n    finally:\\n        pipe.register_modules(text_encoder=encoder)\\n    _after=torch.cuda.Event(enable_timing=True);_after.record(torch.cuda.current_stream(0))\\n    torch.cuda.synchronize(0)\\n    _denoise=_before.elapsed_time(_step_events[-1])/1000 if _step_events else None\\n    _decode=_step_events[-1].elapsed_time(_after)/1000 if _step_events else None\\n    stats = {\\\'model_id\\\':\\\'qwen-image-2.1\\\',\\\'denoising_seconds\\\':round(_denoise,3) if _denoise is not None else None,\\n             \\\'decode_seconds\\\':round(_decode,3) if _decode is not None else None,\\n             \\\'denoising_steps\\\':len(_step_events),\\n             \\\'milliseconds_per_step\\\':round(_denoise*1000/len(_step_events),2) if _step_events else None,\\n             \\\'attention_backend\\\':\\\'sdpa\\\',\\\'parallel_strategy\\\':\\\'encoder_split\\\',\\n             \\\'encoding_seconds\\\': round(encoded-started, 2),\\n             \\\'generation_seconds\\\': round(time.perf_counter()-encoded, 2),\\n             \\\'width\\\': width, \\\'height\\\': height, \\\'steps\\\': steps, \\\'seed\\\': seed,\\n             \\\'peak_gpu_gib\\\': [round(torch.cuda.max_memory_allocated(i)/2**30, 2) for i in range(2)]}\\n    return result, stats\\n\', \'<qwen-model-bootstrap>\', \'exec\'), globals())\n    _existing_state = {\'dataset\': _expected_dataset, \'runtime_id\': uuid.uuid4().hex,\n                       \'load_count\': (_existing_state or {}).get(\'load_count\', 0) + 1,\n                       \'generations\': 0, \'adopted\': False, \'loaded_this_request\': True}\n_COMFY_QWEN_STATE = _existing_state\nif globals().get(\'_QWEN_CACHE_HANDLE\') and globals().get(\'_QWEN_CACHE_TOKEN\'):\n    save_private_cache(ASSET_DIR, encoder)\nprint(\'COMFY_SESSION_READY:\' + json.dumps({\'runtime_id\': _COMFY_QWEN_STATE[\'runtime_id\'],\n    \'load_count\': _COMFY_QWEN_STATE[\'load_count\'], \'reused\': not _COMFY_QWEN_STATE[\'loaded_this_request\']}), flush=True)\n', 'z-image-turbo': '\n_CC_REQUESTED_MODEL = \'z-image-turbo\'\n_CC_PREVIOUS_MODEL = globals().get(\'_CC_MODEL_ID\',\'qwen-image-2.1\')\nif _CC_PREVIOUS_MODEL != _CC_REQUESTED_MODEL:\n    if (globals().get(\'pipe\') is not None or globals().get(\'encoder\') is not None) and not (globals().get(\'_COMFY_QWEN_STATE\') or globals().get(\'_CC_MANAGED_MODEL\')):\n        raise RuntimeError(\'Existing unmanaged model; select the managed notebook kernel.\')\n    # Do not retain aliases to an old pipeline or its GPU modules during collection.\n    for _cc_key in (\'pipe\',\'encoder\',\'transformer\',\'vae\',\'processor\',\'scheduler\',\'generate_image\',\'_COMFY_QWEN_STATE\',\'_CC_NEGATIVE_CACHE\',\'_CC_PROMPT_CACHE\',\'_value\',\'_module\',\'_decoded\'):\n        globals().pop(_cc_key,None)\n    import gc\n    gc.collect()\n    try:\n        import torch\n        torch.cuda.empty_cache()\n    except ImportError:pass\n_CC_MODEL_ID = _CC_REQUESTED_MODEL\n_CC_MANAGED_MODEL = True\n\nimport json,uuid\n_CC_MODEL_PROFILE = {\'id\': \'z-image-turbo\', \'name\': \'Z-Image Turbo · Efficient\', \'default_steps\': 9, \'fixed_steps\': None, \'cache_slug\': \'cc-media-z-image-turbo\', \'family\': \'zimage\'}\n_CC_MODEL_SOURCES = [(\'Tongyi-MAI/Z-Image-Turbo\', \'f332072aa78be7aecdf3ee76d5c247082da564a6\')]\n# JSON arrays are used in saved manifests, including revision tuples.\n_CC_MODEL_SOURCES = json.loads(json.dumps(_CC_MODEL_SOURCES))\nexec(compile(\'"""Download public HF weights with resumable transfers and real byte progress."""\\nimport hashlib\\nimport json\\nimport os\\nimport shutil\\nimport time\\nfrom pathlib import Path, PurePosixPath\\nfrom urllib.parse import quote\\nimport requests\\n\\nBASE_REPO = \\\'Qwen/Qwen-Image-2.1\\\'\\nGGUF_REPO = \\\'azukivc/Qwen-Image-2.1-GGUF\\\'\\nBASE_REVISION = \\\'d26bb61231c349cf6b7896fa83353113880e1ba3\\\'\\nGGUF_REVISION = \\\'4e43fd824763d3d24233630c343462592f68ca49\\\'\\n\\n\\ndef emit_progress(phase, message, done=None, total=None, filename=None):\\n    event = {\\\'phase\\\': phase, \\\'message\\\': message}\\n    if done is not None: event[\\\'downloaded_bytes\\\'] = done\\n    if total: event.update(total_bytes=total, percent=round(min(100, 100*done/total), 1))\\n    if filename: event[\\\'file\\\'] = filename\\n    callback = globals().get(\\\'_QWEN_PROGRESS_CALLBACK\\\')\\n    if callback: callback(event)\\n    print(\\\'COMFY_PROGRESS:\\\' + json.dumps(event, ensure_ascii=False), flush=True)\\n\\n\\ndef repository_manifest(repo, session=None, revision=None):\\n    http = session or requests.Session()\\n    endpoint = \\\'https://huggingface.co/api/models/\\\' + repo\\n    if revision: endpoint += \\\'/revision/\\\' + revision\\n    response = http.get(endpoint, params={\\\'blobs\\\':\\\'true\\\'}, timeout=(20,60))\\n    response.raise_for_status()\\n    data = response.json()\\n    if data.get(\\\'private\\\') or data.get(\\\'gated\\\'):\\n        raise RuntimeError(\\\'默认权重必须能够公开下载，当前来源需要额外授权。\\\')\\n    return data\\n\\n\\ndef safe_relative(name):\\n    path = PurePosixPath(name)\\n    if path.is_absolute() or \\\'..\\\' in path.parts or \\\'\\\\\\\\\\\' in name:\\n        raise ValueError(\\\'Invalid model file path\\\')\\n    return path\\n\\n\\ndef download_file(http, url, destination, expected_size, expected_sha=None, progress=None, verify_progress=None):\\n    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)\\n    partial = destination.with_name(destination.name + \\\'.partial\\\')\\n    receipt = destination.with_name(destination.name + \\\'.verified.json\\\')\\n    if destination.is_file() and destination.stat().st_size == expected_size:\\n        try:\\n            stamp = json.loads(receipt.read_text())\\n            if stamp == {\\\'size\\\':expected_size, \\\'sha256\\\':expected_sha}:\\n                if progress: progress(expected_size)\\n                return\\n        except (OSError,ValueError): pass\\n        # Existing files without a verification receipt must still be checked.\\n        partial.unlink(missing_ok=True); destination.replace(partial)\\n    for attempt in range(3):\\n        try:\\n            offset = partial.stat().st_size if partial.exists() else 0\\n            if expected_size and offset > expected_size:\\n                partial.unlink(); offset = 0\\n            if offset != expected_size or not partial.exists():\\n                headers = {\\\'Range\\\':f\\\'bytes={offset}-\\\'} if offset else {}\\n                with http.get(url, headers=headers, stream=True, timeout=(20,60)) as response:\\n                    response.raise_for_status()\\n                    if offset and response.status_code == 206:\\n                        if not response.headers.get(\\\'Content-Range\\\',\\\'\\\').startswith(f\\\'bytes {offset}-\\\'):\\n                            raise ValueError(\\\'Invalid download resume range\\\')\\n                    else:\\n                        offset = 0\\n                    with partial.open(\\\'ab\\\' if offset else \\\'wb\\\') as output:\\n                        for chunk in response.iter_content(chunk_size=4*1024*1024):\\n                            if chunk:\\n                                output.write(chunk); offset += len(chunk)\\n                                if progress: progress(offset)\\n            if expected_size and partial.stat().st_size != expected_size:\\n                raise OSError(\\\'Incomplete model download\\\')\\n            if expected_sha:\\n                digest = hashlib.sha256()\\n                with partial.open(\\\'rb\\\') as content:\\n                    verified = 0\\n                    for chunk in iter(lambda:content.read(16*1024*1024), b\\\'\\\'):\\n                        digest.update(chunk); verified += len(chunk)\\n                        if verify_progress: verify_progress(verified)\\n                if digest.hexdigest() != expected_sha:\\n                    partial.unlink(); raise ValueError(\\\'Model checksum mismatch\\\')\\n            partial.replace(destination)\\n            receipt.write_text(json.dumps({\\\'size\\\':expected_size,\\\'sha256\\\':expected_sha}))\\n            if progress: progress(destination.stat().st_size)\\n            return\\n        except (requests.RequestException, OSError, ValueError):\\n            if attempt == 2: raise\\n            time.sleep(2*(attempt+1))\\n\\n\\ndef prepare_hf_assets(folder, http=None):\\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\\n    http = http or requests.Session()\\n    emit_progress(\\\'manifest\\\', \\\'正在从 Hugging Face 获取模型文件清单。\\\')\\n    base = repository_manifest(BASE_REPO, http, BASE_REVISION)\\n    quant = repository_manifest(GGUF_REPO, http, GGUF_REVISION)\\n    records = []\\n    for item in base[\\\'siblings\\\']:\\n        name = item[\\\'rfilename\\\']\\n        if name in {\\\'model_index.json\\\',\\\'LICENSE\\\',\\\'transformer/config.json\\\'} or any(name.startswith(p+\\\'/\\\') for p in [\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'text_encoder\\\']):\\n            if name.endswith(\\\'.json\\\') or name.endswith(\\\'.safetensors\\\') or name.endswith(\\\'.txt\\\') or name.endswith(\\\'.model\\\') or name.endswith(\\\'.jinja\\\') or name.endswith(\\\'.tiktoken\\\') or name == \\\'LICENSE\\\':\\n                records.append((BASE_REPO,base[\\\'sha\\\'],item,name))\\n    candidates = [f for f in quant[\\\'siblings\\\'] if f[\\\'rfilename\\\'].lower().endswith(\\\'q4_k_m.gguf\\\')]\\n    if len(candidates) != 1: raise RuntimeError(\\\'兼容的 Q4_K_M 权重清单不明确。\\\')\\n    records.append((GGUF_REPO,quant[\\\'sha\\\'],candidates[0],\\\'transformer-q4.gguf\\\'))\\n    if not any(name.startswith(\\\'text_encoder/\\\') and name.endswith(\\\'.safetensors\\\') for _,_,_,name in records):\\n        raise RuntimeError(\\\'模型清单缺少文本编码器权重。\\\')\\n    total = sum(int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0) for _,_,item,_ in records)\\n    cached=True\\n    for _,_,item,name in records:\\n        safe_relative(name)\\n        size=int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        target=folder/name\\n        try:\\n            receipt=json.loads(target.with_name(target.name+\\\'.verified.json\\\').read_text())\\n            if not target.is_file() or target.stat().st_size!=size or receipt!={\\\'size\\\':size,\\\'sha256\\\':item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')}:\\n                cached=False;break\\n        except (OSError,ValueError):cached=False;break\\n    if cached:\\n        emit_progress(\\\'cached\\\',\\\'已找到完整模型缓存，无需重新下载。\\\',total,total or None)\\n        return folder\\n    done = 0; last = [0.0]\\n    for repo,revision,item,name in records:\\n        safe_relative(name)\\n        size = int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        sha = item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')\\n        start = done\\n        def progress(value, name=name, start=start):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'download\\\',\\\'首次准备：正在下载模型权重。\\\',start+value,total or None,name)\\n                last[0]=now\\n        url=\\\'https://huggingface.co/\\\'+repo+\\\'/resolve/\\\'+revision+\\\'/\\\'+quote(item[\\\'rfilename\\\'],safe=\\\'/\\\')\\n        def verify_progress(value, name=name, size=size):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'verify\\\',\\\'正在校验下载的权重。\\\',value,size,name);last[0]=now\\n        download_file(http,url,folder/name,size,sha,progress,verify_progress)\\n        done += size\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps({\\\'base_repo\\\':BASE_REPO,\\\'base_revision\\\':base[\\\'sha\\\'],\\\'gguf_repo\\\':GGUF_REPO,\\\'gguf_revision\\\':quant[\\\'sha\\\'],\\\'download_bytes\\\':total},indent=2))\\n    emit_progress(\\\'download\\\',\\\'模型权重下载完成。\\\',done,total or None)\\n    return folder\\n\\n\\ndef copy_assets(source, target):\\n    source,target=Path(source),Path(target)\\n    files=[p for p in source.rglob(\\\'*\\\') if p.is_file()]\\n    total=sum(p.stat().st_size for p in files); done=0; last=0\\n    for path in files:\\n        destination=target/path.relative_to(source); destination.parent.mkdir(parents=True,exist_ok=True)\\n        with path.open(\\\'rb\\\') as src, destination.open(\\\'wb\\\') as dst:\\n            for chunk in iter(lambda:src.read(8*1024*1024),b\\\'\\\'):\\n                dst.write(chunk);done+=len(chunk)\\n                now=time.monotonic()\\n                if now-last>=.8:\\n                    emit_progress(\\\'copy\\\',\\\'正在准备已缓存的模型文件。\\\',done,total,path.name);last=now\\n    emit_progress(\\\'copy\\\',\\\'模型文件准备完成。\\\',done,total)\\n\\n"""Private, create-only Kaggle cache of prepared INT8/Q4 model assets.\\n\\nExecuted in the user\\\'s Kaggle kernel. Credentials are passed at connection time,\\nused only for Kaggle operations, and removed by the caller\\\'s finally block.\\n"""\\nimport contextlib\\nimport tempfile\\nimport threading\\nfrom concurrent.futures import ThreadPoolExecutor\\n\\nCACHE_VERSION = 1\\n\\n\\ndef cache_file_hashes(folder):\\n    hashes = {}\\n    for file in Path(folder).rglob(\\\'*\\\'):\\n        if file.is_file():\\n            with file.open(\\\'rb\\\') as content:\\n                hashes[file.relative_to(folder).as_posix()] = hashlib.file_digest(content,\\\'sha256\\\').hexdigest()\\n    return hashes\\n\\n\\n@contextlib.contextmanager\\ndef cache_credentials(handle, token):\\n    owner = handle.split(\\\'/\\\')[0]\\n    values = {\\\'KAGGLE_USERNAME\\\': owner,\\\'KAGGLE_API_TOKEN\\\':None,\\\'KAGGLE_KEY\\\':None}\\n    if token.startswith(\\\'KGAT_\\\'): values[\\\'KAGGLE_API_TOKEN\\\'] = token\\n    else: values[\\\'KAGGLE_KEY\\\'] = token\\n    old = {name: os.environ.get(name) for name in values}\\n    try:\\n        for name,value in values.items():\\n            if value is None: os.environ.pop(name,None)\\n            else: os.environ[name]=value\\n        yield\\n    finally:\\n        for name, value in old.items():\\n            if value is None: os.environ.pop(name, None)\\n            else: os.environ[name] = value\\n\\n\\ndef cache_metadata(handle, token):\\n    owner, slug = handle.split(\\\'/\\\')\\n    with requests.Session() as http:\\n        if token.startswith(\\\'KGAT_\\\'): http.headers[\\\'Authorization\\\'] = \\\'Bearer \\\' + token\\n        else: http.auth = (owner, token)\\n        response = http.get(\\\'https://www.kaggle.com/api/v1/datasets/view/\\\'+owner+\\\'/\\\'+slug, timeout=(20,60))\\n        if response.status_code in (403,404):\\n            # Kaggle hides missing private slugs behind 403. Verify absence in\\n            # the authenticated owner listing; a permission failure stays fatal.\\n            for page in range(1,101):\\n                listing = http.get(\\\'https://www.kaggle.com/api/v1/datasets/list\\\',\\n                    params={\\\'user\\\':owner,\\\'search\\\':slug,\\\'page\\\':page}, timeout=(20,60))\\n                listing.raise_for_status()\\n                records = listing.json()\\n                if not isinstance(records,list): raise RuntimeError(\\\'Invalid dataset listing\\\')\\n                if any(item.get(\\\'ref\\\') == handle for item in records):\\n                    raise RuntimeError(\\\'Existing cache is inaccessible\\\')\\n                if len(records)<20: return None\\n            raise RuntimeError(\\\'Cannot confirm cache absence\\\')\\n        response.raise_for_status()\\n        data = response.json()\\n        if data.get(\\\'isPrivate\\\') is not True:\\n            raise RuntimeError(\\\'模型缓存必须为私有 Dataset；没有修改现有 Dataset。\\\')\\n        return data\\n\\n\\ndef validate_cache_manifest(manifest):\\n    generic=manifest.get(\\\'studio_model_id\\\') is not None\\n    if generic:\\n        if (manifest.get(\\\'studio_cache_version\\\')!=2 or manifest[\\\'studio_model_id\\\']!=globals().get(\\\'_CC_MODEL_ID\\\')\\n            or manifest.get(\\\'model_sources\\\')!=globals().get(\\\'_CC_MODEL_SOURCES\\\')):\\n            raise RuntimeError(\\\'所选模型缓存版本不兼容。\\\')\\n    elif (manifest.get(\\\'studio_cache_version\\\') != CACHE_VERSION\\n        or manifest.get(\\\'base_revision\\\') != BASE_REVISION\\n        or manifest.get(\\\'gguf_revision\\\') != GGUF_REVISION):\\n        raise RuntimeError(\\\'同名 Dataset 不是兼容的本程序缓存，没有覆盖它。\\\')\\n    files = manifest.get(\\\'cache_files\\\', {})\\n    if not files or (not generic and \\\'transformer-q4.gguf\\\' not in files) or (generic and not any(name.endswith(\\\'.safetensors\\\') for name in files)):\\n        raise RuntimeError(\\\'模型缓存文件清单不完整。\\\')\\n    for name, size in files.items():\\n        safe_relative(name)\\n        if not isinstance(size,int) or size<0: raise RuntimeError(\\\'Invalid cached file size\\\')\\n    return manifest\\n\\n\\ndef validate_cache(folder):\\n    folder = Path(folder)\\n    manifest = validate_cache_manifest(json.loads((folder/\\\'asset-sources.json\\\').read_text()))\\n    for name, size in manifest[\\\'cache_files\\\'].items():\\n        file = folder/name\\n        if not file.is_file() or file.stat().st_size != size:\\n            raise RuntimeError(\\\'模型缓存文件不完整。\\\')\\n    return manifest\\n\\n\\ndef cache_http(handle, token):\\n    http = requests.Session()\\n    if token.startswith(\\\'KGAT_\\\'): http.headers[\\\'Authorization\\\'] = \\\'Bearer \\\'+token\\n    else: http.auth = (handle.split(\\\'/\\\')[0],token)\\n    return http\\n\\n\\ndef cache_file_url(handle, version, name):\\n    safe_relative(name)\\n    return (\\\'https://www.kaggle.com/api/v1/datasets/download/\\\'+handle+\\\'/\\\'+quote(name,safe=\\\'\\\')+\\n        \\\'?raw=true&dataset_version_number=\\\'+str(int(version)))\\n\\n\\ndef read_cache_manifest(http, handle, version):\\n    response = http.get(cache_file_url(handle,version,\\\'asset-sources.json\\\'),timeout=(20,60))\\n    response.raise_for_status()\\n    return validate_cache_manifest(response.json())\\n\\n\\ndef download_cache_assets(folder, handle, token, metadata):\\n    folder = Path(folder); folder.mkdir(parents=True,exist_ok=True)\\n    version = metadata[\\\'currentVersionNumber\\\']\\n    with cache_http(handle,token) as http:\\n        manifest = read_cache_manifest(http,handle,version)\\n    total = sum(manifest[\\\'cache_files\\\'].values()); received={}; last=[0.0]\\n    progress_lock=threading.Lock()\\n    def transfer(item):\\n        name,size=item\\n        def progress(value):\\n            with progress_lock:\\n                received[name]=value\\n                now=time.monotonic()\\n                if now-last[0]>=.8 or value==size:\\n                    emit_progress(\\\'cache-download\\\',\\\'正在拉取私有 Dataset 模型缓存。\\\',sum(received.values()),total,name)\\n                    last[0]=now\\n        with cache_http(handle,token) as http:\\n            download_file(http,cache_file_url(handle,version,name),folder/name,size,\\n                manifest.get(\\\'cache_sha256\\\',{}).get(name),progress)\\n    # Independent resumable streams avoid serializing all multi-GB shards.\\n    files=sorted(manifest[\\\'cache_files\\\'].items(),key=lambda item:item[1],reverse=True)\\n    with ThreadPoolExecutor(max_workers=4) as pool:\\n        list(pool.map(transfer,files))\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps(manifest,indent=2))\\n    validate_cache(folder)\\n    return folder\\n\\n\\ndef prepare_auto_assets(folder):\\n    handle, token = globals().get(\\\'_QWEN_CACHE_HANDLE\\\'), globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'disabled\\\'\\n    globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\'] = handle\\n    if handle and token:\\n        try:\\n            emit_progress(\\\'cache-check\\\', \\\'正在检查你的私有模型缓存。\\\')\\n            metadata = cache_metadata(handle, token)\\n            if metadata:\\n                emit_progress(\\\'cache-download\\\', \\\'正在拉取私有 Dataset 模型缓存。\\\')\\n                download_cache_assets(folder, handle, token, metadata)\\n                globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n                globals()[\\\'_QWEN_RESOLVED_SOURCE\\\'] = \\\'kaggle-cache\\\'\\n                return Path(folder), False\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'missing\\\'\\n        except Exception:\\n            # A metadata/auth/compatibility error must never permit overwriting\\n            # an existing dataset. Generation can still use the public weights.\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'unavailable\\\'\\n            emit_progress(\\\'cache-warning\\\', \\\'私有缓存暂不可用，本次从 Hugging Face 准备；没有修改已有 Dataset。\\\')\\n    globals()[\\\'_QWEN_RESOLVED_SOURCE\\\'] = \\\'huggingface\\\'\\n    return prepare_hf_assets(folder), True\\n\\n\\ndef save_private_cache(folder, encoder):\\n    handle, token = globals().get(\\\'_QWEN_CACHE_HANDLE\\\'), globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    status = globals().get(\\\'_QWEN_CACHE_STATUS\\\', \\\'missing\\\') if globals().get(\\\'_QWEN_CACHE_STATUS_HANDLE\\\') == handle else \\\'missing\\\'\\n    globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\'] = handle\\n    if not handle or not token or status in (\\\'saved\\\',\\\'unavailable\\\'): return\\n    try:\\n        emit_progress(\\\'cache-check\\\', \\\'正在确认私有缓存状态。\\\')\\n        metadata = cache_metadata(handle, token)\\n        if metadata:\\n            with cache_http(handle,token) as http:\\n                read_cache_manifest(http,handle,metadata[\\\'currentVersionNumber\\\'])\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n            return\\n        folder = Path(folder)\\n        emit_progress(\\\'cache-prepare\\\', \\\'正在保存 INT8 编码器和 Q4 权重，首次需要额外等待。\\\')\\n        with tempfile.TemporaryDirectory(prefix=\\\'qwen-private-cache-\\\', dir=\\\'/kaggle/temp\\\') as temporary:\\n            bundle = Path(temporary)\\n            generic=globals().get(\\\'_CC_MODEL_ID\\\',\\\'qwen-image-2.1\\\')!=\\\'qwen-image-2.1\\\'\\n            if generic:\\n                def link_or_copy(src,dst):\\n                    try:os.link(src,dst)\\n                    except OSError:shutil.copy2(src,dst)\\n                shutil.copytree(folder,bundle,dirs_exist_ok=True,copy_function=link_or_copy,\\n                    ignore=shutil.ignore_patterns(\\\'*.partial\\\',\\\'*.verified.json\\\',\\\'asset-sources.json\\\'))\\n            for name in (() if generic else (\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'transformer\\\')):\\n                shutil.copytree(folder/name, bundle/name, ignore=shutil.ignore_patterns(\\\'*.partial\\\',\\\'*.verified.json\\\'))\\n            for name in (() if generic else (\\\'transformer-q4.gguf\\\',\\\'model_index.json\\\',\\\'LICENSE\\\')):\\n                if (folder/name).exists(): shutil.copy2(folder/name, bundle/name)\\n            if not generic:encoder.save_pretrained(bundle/\\\'text_encoder\\\', safe_serialization=True, max_shard_size=\\\'4GB\\\')\\n            manifest = json.loads((folder/\\\'asset-sources.json\\\').read_text())\\n            manifest.update(studio_cache_version=2 if generic else CACHE_VERSION,\\n                cache_files={p.relative_to(bundle).as_posix():p.stat().st_size for p in bundle.rglob(\\\'*\\\') if p.is_file()},\\n                cache_sha256=cache_file_hashes(bundle))\\n            (bundle/\\\'asset-sources.json\\\').write_text(json.dumps(manifest, indent=2))\\n            validate_cache(bundle)\\n            emit_progress(\\\'cache-upload\\\', \\\'正在上传私有 Dataset；Kaggle 将显示文件上传进度。\\\')\\n            # Use KaggleHub\\\'s create-only primitive: its public upload helper\\n            # creates a new version on conflict, which could overwrite user data.\\n            from kagglehub import datasets\\n            api = datasets.dataset_upload.__globals__\\n            create = datasets.create_dataset_or_version.__globals__[\\\'_create_dataset\\\']\\n            with cache_credentials(handle, token):\\n                uploads = api[\\\'upload_files_and_directories\\\'](str(bundle), item_type=api[\\\'ApiBlobType\\\'].DATASET,\\n                    ignore_patterns=api[\\\'normalize_patterns\\\'](default=api[\\\'DEFAULT_IGNORE_PATTERNS\\\'], additional=None))\\n                create(api[\\\'parse_dataset_handle\\\'](handle), uploads)\\n            # The create primitive explicitly sends is_private=True. Confirm\\n            # privacy again from the server before reporting success.\\n            emit_progress(\\\'cache-processing\\\', \\\'文件已上传，Kaggle 正在处理私有 Dataset。\\\')\\n            deadline = time.monotonic()+180\\n            while not cache_metadata(handle, token):\\n                if time.monotonic() >= deadline: raise RuntimeError(\\\'Dataset metadata not ready\\\')\\n                time.sleep(5)\\n        globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n        emit_progress(\\\'cache-saved\\\', \\\'私有模型缓存已保存，下次新会话优先拉取，无需重新下载或量化。\\\')\\n    except Exception as error:\\n        globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'failed\\\'\\n        globals()[\\\'_QWEN_CACHE_FAILURE_TYPE\\\'] = type(error).__name__\\n        emit_progress(\\\'cache-warning\\\', \\\'私有缓存保存未完成；本次仍可正常生图，重新连接后会再次检查。\\\')\\n\\n"""Public, pinned, resumable model downloads. Executed after asset_manager.py."""\\n\\ndef model_asset_records(model_id, sources, http):\\n    records=[]\\n    for index,(repo,revision) in enumerate(sources):\\n        manifest=repository_manifest(repo,http,revision)\\n        if manifest.get(\\\'sha\\\')!=revision:raise RuntimeError(\\\'Model revision mismatch\\\')\\n        for item in manifest[\\\'siblings\\\']:\\n            name=item[\\\'rfilename\\\'];safe_relative(name)\\n            keep=False\\n            if model_id==\\\'z-image-turbo\\\':\\n                keep=(name in (\\\'model_index.json\\\',\\\'LICENSE\\\',\\\'LICENSE.txt\\\',\\\'README.md\\\') or\\n                      name.split(\\\'/\\\')[0] in (\\\'text_encoder\\\',\\\'tokenizer\\\',\\\'transformer\\\',\\\'vae\\\',\\\'scheduler\\\'))\\n                keep=keep and name.endswith((\\\'.json\\\',\\\'.safetensors\\\',\\\'.txt\\\',\\\'.model\\\',\\\'.jinja\\\',\\\'.tiktoken\\\',\\\'.md\\\'))\\n            elif index==0:\\n                # Lightning replaces the base UNet; do not download a second one.\\n                parts=name.split(\\\'/\\\')\\n                keep=name in (\\\'model_index.json\\\',\\\'LICENSE.md\\\',\\\'README.md\\\',\\\'unet/config.json\\\')\\n                if parts[0] in (\\\'text_encoder\\\',\\\'text_encoder_2\\\',\\\'tokenizer\\\',\\\'tokenizer_2\\\',\\\'scheduler\\\',\\\'vae\\\'):\\n                    keep=name.endswith((\\\'.json\\\',\\\'.txt\\\',\\\'.model\\\',\\\'.jinja\\\')) or name.endswith(\\\'.fp16.safetensors\\\')\\n            else:keep=name in (\\\'sdxl_lightning_4step_unet.safetensors\\\',\\\'LICENSE.md\\\',\\\'README.md\\\')\\n            if keep:records.append((repo,revision,item,(\\\'base/\\\' if index==0 else \\\'lightning/\\\')+name if model_id==\\\'sdxl-lightning\\\' else name))\\n    if not any(name.endswith(\\\'.safetensors\\\') for _,_,_,name in records):raise RuntimeError(\\\'Model weights missing\\\')\\n    return records\\n\\n\\ndef prepare_model_hf_assets(folder):\\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\\n    with requests.Session() as http:\\n        emit_progress(\\\'manifest\\\',\\\'正在获取所选模型清单。\\\')\\n        records=model_asset_records(_CC_MODEL_ID,_CC_MODEL_SOURCES,http)\\n        total=sum(int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0) for _,_,item,_ in records)\\n        done=0;last=[0.0]\\n        for repo,revision,item,name in records:\\n            size=int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n            if size<=0:raise RuntimeError(\\\'Model file size missing\\\')\\n            start=done\\n            def progress(value,name=name,start=start,size=size):\\n                now=time.monotonic()\\n                if now-last[0]>.8 or value==size:\\n                    emit_progress(\\\'download\\\',\\\'正在下载所选模型。\\\',start+value,total,name);last[0]=now\\n            url=\\\'https://huggingface.co/\\\'+repo+\\\'/resolve/\\\'+revision+\\\'/\\\'+quote(item[\\\'rfilename\\\'],safe=\\\'/\\\')\\n            download_file(http,url,folder/name,size,item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\'),progress)\\n            done+=size\\n    manifest={\\\'studio_model_id\\\':_CC_MODEL_ID,\\\'model_sources\\\':_CC_MODEL_SOURCES,\\\'download_bytes\\\':total}\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps(manifest,indent=2))\\n    emit_progress(\\\'download\\\',\\\'所选模型下载完成。\\\',total,total)\\n    return folder\\n\\n\\ndef prepare_selected_model_assets(folder):\\n    handle,token=globals().get(\\\'_QWEN_CACHE_HANDLE\\\'),globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'disabled\\\';globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\']=handle\\n    if handle and token:\\n        try:\\n            emit_progress(\\\'cache-check\\\',\\\'正在检查所选模型的私有缓存。\\\')\\n            metadata=cache_metadata(handle,token)\\n            if metadata:\\n                download_cache_assets(folder,handle,token,metadata)\\n                globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'saved\\\';globals()[\\\'_QWEN_RESOLVED_SOURCE\\\']=\\\'kaggle-cache\\\'\\n                return Path(folder)\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'missing\\\'\\n        except Exception:\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'unavailable\\\'\\n            emit_progress(\\\'cache-warning\\\',\\\'正在从 Hugging Face 准备所选模型。\\\')\\n    globals()[\\\'_QWEN_RESOLVED_SOURCE\\\']=\\\'huggingface\\\'\\n    return prepare_model_hf_assets(folder)\\n\',\'<cc-model-assets>\',\'exec\'),globals())\n_existing_state=globals().get(\'_COMFY_QWEN_STATE\')\nif not (_existing_state and globals().get(\'pipe\') is not None and callable(globals().get(\'generate_image\'))):\n    if False:\n        import subprocess,sys\n        subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'bitsandbytes==0.50.2\',\'safetensors\',\'diffusers>=0.36,<0.43\',\'transformers>=4.51,<6\',\'accelerate>=1.2\'],check=True,timeout=900)\n    for _cc_partial in (\'pipe\',\'encoder\',\'transformer\',\'vae\',\'processor\',\'scheduler\',\'generate_image\'):\n        globals().pop(_cc_partial,None)\n    import gc\n    gc.collect()\n    import torch\n    torch.cuda.empty_cache()\n    exec(compile(\'"""Additional model loaders. The caller embeds pinned catalog and asset helpers."""\\nimport gc\\nimport time\\nimport torch\\n\\nassert torch.cuda.device_count()>=2,\\\'Select GPU T4 x2.\\\'\\nASSET_DIR=Path(\\\'/kaggle/temp/cc-media-models\\\')/_CC_MODEL_ID\\nstarted=time.perf_counter()\\nprepare_selected_model_assets(ASSET_DIR)\\n_assets_ready=time.perf_counter()\\nemit_progress(\\\'loading\\\',\\\'正在加载所选模型。\\\')\\nif _CC_MODEL_ID==\\\'z-image-turbo\\\':\\n    from diffusers import ZImagePipeline,ZImageTransformer2DModel,AutoencoderKL\\n    from transformers import Qwen3Model,BitsAndBytesConfig\\n    encoder=Qwen3Model.from_pretrained(ASSET_DIR/\\\'text_encoder\\\',torch_dtype=torch.float16,\\n        quantization_config=BitsAndBytesConfig(load_in_8bit=True),device_map={\\\'\\\':1},local_files_only=True)\\n    transformer=ZImageTransformer2DModel.from_pretrained(ASSET_DIR/\\\'transformer\\\',torch_dtype=torch.float16,local_files_only=True).to(\\\'cuda:0\\\')\\n    vae=AutoencoderKL.from_pretrained(ASSET_DIR/\\\'vae\\\',torch_dtype=torch.float32,local_files_only=True).to(\\\'cuda:0\\\')\\n    pipe=ZImagePipeline.from_pretrained(ASSET_DIR,torch_dtype=torch.float16,text_encoder=encoder,\\n        transformer=transformer,vae=vae,local_files_only=True)\\nelse:\\n    from diffusers import StableDiffusionXLPipeline,UNet2DConditionModel,EulerDiscreteScheduler\\n    from safetensors.torch import load_file\\n    transformer=UNet2DConditionModel.from_config(ASSET_DIR/\\\'base\\\',subfolder=\\\'unet\\\')\\n    transformer.load_state_dict(load_file(ASSET_DIR/\\\'lightning/sdxl_lightning_4step_unet.safetensors\\\',device=\\\'cpu\\\'))\\n    transformer=transformer.to(device=\\\'cuda:0\\\',dtype=torch.float16)\\n    pipe=StableDiffusionXLPipeline.from_pretrained(ASSET_DIR/\\\'base\\\',unet=transformer,\\n        torch_dtype=torch.float16,variant=\\\'fp16\\\',local_files_only=True).to(\\\'cuda:0\\\')\\n    pipe.scheduler=EulerDiscreteScheduler.from_config(pipe.scheduler.config,timestep_spacing=\\\'trailing\\\')\\n    encoder=pipe.text_encoder\\nload_stats={\\\'copy_seconds\\\':round(_assets_ready-started,2),\\\'loading_seconds\\\':round(time.perf_counter()-_assets_ready,2)}\\n\\n\\n@torch.inference_mode()\\ndef generate_image(prompt,seed=42,width=512,height=512,steps=None):\\n    steps=_CC_MODEL_PROFILE[\\\'default_steps\\\'] if steps is None else steps\\n    if _CC_MODEL_ID==\\\'sdxl-lightning\\\' and steps!=4:raise ValueError(\\\'SDXL-Lightning requires 4 steps\\\')\\n    for i in range(2):torch.cuda.reset_peak_memory_stats(i)\\n    started=time.perf_counter();encoding_seconds=0\\n    if _CC_MODEL_ID==\\\'z-image-turbo\\\':\\n        positive,_=pipe.encode_prompt(prompt,device=torch.device(\\\'cuda:1\\\'),do_classifier_free_guidance=False)\\n        torch.cuda.synchronize(1);encoding_seconds=time.perf_counter()-started\\n        kwargs={\\\'prompt_embeds\\\':[value.to(\\\'cuda:0\\\') for value in positive],\\\'guidance_scale\\\':0.0}\\n    else:\\n        positive,negative,pooled,negative_pooled=pipe.encode_prompt(prompt=prompt,device=torch.device(\\\'cuda:0\\\'),do_classifier_free_guidance=False)\\n        torch.cuda.synchronize(0);encoding_seconds=time.perf_counter()-started\\n        kwargs={\\\'prompt_embeds\\\':positive,\\\'pooled_prompt_embeds\\\':pooled,\\\'guidance_scale\\\':0.0}\\n    # Explicit denoising/decode timing; CUDA events avoid synchronizing every step.\\n    timings=[]\\n    def record_step(pipeline,index,timestep,callback_kwargs):\\n        event=torch.cuda.Event(enable_timing=True);event.record(torch.cuda.current_stream(0));timings.append(event)\\n        return callback_kwargs\\n    before=torch.cuda.Event(enable_timing=True);before.record(torch.cuda.current_stream(0))\\n    result=pipe(width=width,height=height,num_inference_steps=steps,\\n        generator=torch.Generator(device=\\\'cuda:0\\\').manual_seed(seed),callback_on_step_end=record_step,**kwargs).images[0]\\n    after=torch.cuda.Event(enable_timing=True);after.record(torch.cuda.current_stream(0));torch.cuda.synchronize(0)\\n    denoise=before.elapsed_time(timings[-1])/1000 if timings else None\\n    decode=timings[-1].elapsed_time(after)/1000 if timings else None\\n    return result,{\\\'model_id\\\':_CC_MODEL_ID,\\\'encoding_seconds\\\':round(encoding_seconds,3),\\n        \\\'generation_seconds\\\':round(time.perf_counter()-started-encoding_seconds,3),\\n        \\\'denoising_seconds\\\':round(denoise,3) if denoise is not None else None,\\n        \\\'decode_seconds\\\':round(decode,3) if decode is not None else None,\\n        \\\'denoising_steps\\\':len(timings),\\\'milliseconds_per_step\\\':round(denoise*1000/len(timings),2) if timings else None,\\n        \\\'attention_backend\\\':\\\'sdpa\\\',\\\'parallel_strategy\\\':\\\'encoder_split\\\' if _CC_MODEL_ID==\\\'z-image-turbo\\\' else \\\'single_gpu\\\',\\n        \\\'width\\\':width,\\\'height\\\':height,\\\'steps\\\':steps,\\\'seed\\\':seed,\\n        \\\'peak_gpu_gib\\\':[round(torch.cuda.max_memory_allocated(i)/2**30,2) for i in range(2)]}\\n\',\'<cc-model-runtime>\',\'exec\'),globals())\n    _existing_state={\'dataset\':\'model/\'+_CC_MODEL_ID,\'model_id\':_CC_MODEL_ID,\'runtime_id\':uuid.uuid4().hex,\'load_count\':1,\'generations\':0,\'loaded_this_request\':True}\nelse:\n    _existing_state[\'loaded_this_request\']=False\n_COMFY_QWEN_STATE=_existing_state\nif globals().get(\'_QWEN_CACHE_HANDLE\') and globals().get(\'_QWEN_CACHE_TOKEN\'):\n    save_private_cache(ASSET_DIR,encoder)\nprint(\'COMFY_SESSION_READY:\'+json.dumps({\'runtime_id\':_existing_state[\'runtime_id\'],\'load_count\':_existing_state[\'load_count\'],\'reused\':not _existing_state[\'loaded_this_request\']}),flush=True)\n', 'sdxl-lightning': '\n_CC_REQUESTED_MODEL = \'sdxl-lightning\'\n_CC_PREVIOUS_MODEL = globals().get(\'_CC_MODEL_ID\',\'qwen-image-2.1\')\nif _CC_PREVIOUS_MODEL != _CC_REQUESTED_MODEL:\n    if (globals().get(\'pipe\') is not None or globals().get(\'encoder\') is not None) and not (globals().get(\'_COMFY_QWEN_STATE\') or globals().get(\'_CC_MANAGED_MODEL\')):\n        raise RuntimeError(\'Existing unmanaged model; select the managed notebook kernel.\')\n    # Do not retain aliases to an old pipeline or its GPU modules during collection.\n    for _cc_key in (\'pipe\',\'encoder\',\'transformer\',\'vae\',\'processor\',\'scheduler\',\'generate_image\',\'_COMFY_QWEN_STATE\',\'_CC_NEGATIVE_CACHE\',\'_CC_PROMPT_CACHE\',\'_value\',\'_module\',\'_decoded\'):\n        globals().pop(_cc_key,None)\n    import gc\n    gc.collect()\n    try:\n        import torch\n        torch.cuda.empty_cache()\n    except ImportError:pass\n_CC_MODEL_ID = _CC_REQUESTED_MODEL\n_CC_MANAGED_MODEL = True\n\nimport json,uuid\n_CC_MODEL_PROFILE = {\'id\': \'sdxl-lightning\', \'name\': \'SDXL-Lightning · 4-step Preview\', \'default_steps\': 4, \'fixed_steps\': 4, \'cache_slug\': \'cc-media-sdxl-lightning\', \'family\': \'sdxl\'}\n_CC_MODEL_SOURCES = [(\'stabilityai/stable-diffusion-xl-base-1.0\', \'462165984030d82259a11f4367a4eed129e94a7b\'), (\'ByteDance/SDXL-Lightning\', \'c9a24f48e1c025556787b0c58dd67a091ece2e44\')]\n# JSON arrays are used in saved manifests, including revision tuples.\n_CC_MODEL_SOURCES = json.loads(json.dumps(_CC_MODEL_SOURCES))\nexec(compile(\'"""Download public HF weights with resumable transfers and real byte progress."""\\nimport hashlib\\nimport json\\nimport os\\nimport shutil\\nimport time\\nfrom pathlib import Path, PurePosixPath\\nfrom urllib.parse import quote\\nimport requests\\n\\nBASE_REPO = \\\'Qwen/Qwen-Image-2.1\\\'\\nGGUF_REPO = \\\'azukivc/Qwen-Image-2.1-GGUF\\\'\\nBASE_REVISION = \\\'d26bb61231c349cf6b7896fa83353113880e1ba3\\\'\\nGGUF_REVISION = \\\'4e43fd824763d3d24233630c343462592f68ca49\\\'\\n\\n\\ndef emit_progress(phase, message, done=None, total=None, filename=None):\\n    event = {\\\'phase\\\': phase, \\\'message\\\': message}\\n    if done is not None: event[\\\'downloaded_bytes\\\'] = done\\n    if total: event.update(total_bytes=total, percent=round(min(100, 100*done/total), 1))\\n    if filename: event[\\\'file\\\'] = filename\\n    callback = globals().get(\\\'_QWEN_PROGRESS_CALLBACK\\\')\\n    if callback: callback(event)\\n    print(\\\'COMFY_PROGRESS:\\\' + json.dumps(event, ensure_ascii=False), flush=True)\\n\\n\\ndef repository_manifest(repo, session=None, revision=None):\\n    http = session or requests.Session()\\n    endpoint = \\\'https://huggingface.co/api/models/\\\' + repo\\n    if revision: endpoint += \\\'/revision/\\\' + revision\\n    response = http.get(endpoint, params={\\\'blobs\\\':\\\'true\\\'}, timeout=(20,60))\\n    response.raise_for_status()\\n    data = response.json()\\n    if data.get(\\\'private\\\') or data.get(\\\'gated\\\'):\\n        raise RuntimeError(\\\'默认权重必须能够公开下载，当前来源需要额外授权。\\\')\\n    return data\\n\\n\\ndef safe_relative(name):\\n    path = PurePosixPath(name)\\n    if path.is_absolute() or \\\'..\\\' in path.parts or \\\'\\\\\\\\\\\' in name:\\n        raise ValueError(\\\'Invalid model file path\\\')\\n    return path\\n\\n\\ndef download_file(http, url, destination, expected_size, expected_sha=None, progress=None, verify_progress=None):\\n    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)\\n    partial = destination.with_name(destination.name + \\\'.partial\\\')\\n    receipt = destination.with_name(destination.name + \\\'.verified.json\\\')\\n    if destination.is_file() and destination.stat().st_size == expected_size:\\n        try:\\n            stamp = json.loads(receipt.read_text())\\n            if stamp == {\\\'size\\\':expected_size, \\\'sha256\\\':expected_sha}:\\n                if progress: progress(expected_size)\\n                return\\n        except (OSError,ValueError): pass\\n        # Existing files without a verification receipt must still be checked.\\n        partial.unlink(missing_ok=True); destination.replace(partial)\\n    for attempt in range(3):\\n        try:\\n            offset = partial.stat().st_size if partial.exists() else 0\\n            if expected_size and offset > expected_size:\\n                partial.unlink(); offset = 0\\n            if offset != expected_size or not partial.exists():\\n                headers = {\\\'Range\\\':f\\\'bytes={offset}-\\\'} if offset else {}\\n                with http.get(url, headers=headers, stream=True, timeout=(20,60)) as response:\\n                    response.raise_for_status()\\n                    if offset and response.status_code == 206:\\n                        if not response.headers.get(\\\'Content-Range\\\',\\\'\\\').startswith(f\\\'bytes {offset}-\\\'):\\n                            raise ValueError(\\\'Invalid download resume range\\\')\\n                    else:\\n                        offset = 0\\n                    with partial.open(\\\'ab\\\' if offset else \\\'wb\\\') as output:\\n                        for chunk in response.iter_content(chunk_size=4*1024*1024):\\n                            if chunk:\\n                                output.write(chunk); offset += len(chunk)\\n                                if progress: progress(offset)\\n            if expected_size and partial.stat().st_size != expected_size:\\n                raise OSError(\\\'Incomplete model download\\\')\\n            if expected_sha:\\n                digest = hashlib.sha256()\\n                with partial.open(\\\'rb\\\') as content:\\n                    verified = 0\\n                    for chunk in iter(lambda:content.read(16*1024*1024), b\\\'\\\'):\\n                        digest.update(chunk); verified += len(chunk)\\n                        if verify_progress: verify_progress(verified)\\n                if digest.hexdigest() != expected_sha:\\n                    partial.unlink(); raise ValueError(\\\'Model checksum mismatch\\\')\\n            partial.replace(destination)\\n            receipt.write_text(json.dumps({\\\'size\\\':expected_size,\\\'sha256\\\':expected_sha}))\\n            if progress: progress(destination.stat().st_size)\\n            return\\n        except (requests.RequestException, OSError, ValueError):\\n            if attempt == 2: raise\\n            time.sleep(2*(attempt+1))\\n\\n\\ndef prepare_hf_assets(folder, http=None):\\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\\n    http = http or requests.Session()\\n    emit_progress(\\\'manifest\\\', \\\'正在从 Hugging Face 获取模型文件清单。\\\')\\n    base = repository_manifest(BASE_REPO, http, BASE_REVISION)\\n    quant = repository_manifest(GGUF_REPO, http, GGUF_REVISION)\\n    records = []\\n    for item in base[\\\'siblings\\\']:\\n        name = item[\\\'rfilename\\\']\\n        if name in {\\\'model_index.json\\\',\\\'LICENSE\\\',\\\'transformer/config.json\\\'} or any(name.startswith(p+\\\'/\\\') for p in [\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'text_encoder\\\']):\\n            if name.endswith(\\\'.json\\\') or name.endswith(\\\'.safetensors\\\') or name.endswith(\\\'.txt\\\') or name.endswith(\\\'.model\\\') or name.endswith(\\\'.jinja\\\') or name.endswith(\\\'.tiktoken\\\') or name == \\\'LICENSE\\\':\\n                records.append((BASE_REPO,base[\\\'sha\\\'],item,name))\\n    candidates = [f for f in quant[\\\'siblings\\\'] if f[\\\'rfilename\\\'].lower().endswith(\\\'q4_k_m.gguf\\\')]\\n    if len(candidates) != 1: raise RuntimeError(\\\'兼容的 Q4_K_M 权重清单不明确。\\\')\\n    records.append((GGUF_REPO,quant[\\\'sha\\\'],candidates[0],\\\'transformer-q4.gguf\\\'))\\n    if not any(name.startswith(\\\'text_encoder/\\\') and name.endswith(\\\'.safetensors\\\') for _,_,_,name in records):\\n        raise RuntimeError(\\\'模型清单缺少文本编码器权重。\\\')\\n    total = sum(int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0) for _,_,item,_ in records)\\n    cached=True\\n    for _,_,item,name in records:\\n        safe_relative(name)\\n        size=int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        target=folder/name\\n        try:\\n            receipt=json.loads(target.with_name(target.name+\\\'.verified.json\\\').read_text())\\n            if not target.is_file() or target.stat().st_size!=size or receipt!={\\\'size\\\':size,\\\'sha256\\\':item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')}:\\n                cached=False;break\\n        except (OSError,ValueError):cached=False;break\\n    if cached:\\n        emit_progress(\\\'cached\\\',\\\'已找到完整模型缓存，无需重新下载。\\\',total,total or None)\\n        return folder\\n    done = 0; last = [0.0]\\n    for repo,revision,item,name in records:\\n        safe_relative(name)\\n        size = int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n        sha = item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\')\\n        start = done\\n        def progress(value, name=name, start=start):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'download\\\',\\\'首次准备：正在下载模型权重。\\\',start+value,total or None,name)\\n                last[0]=now\\n        url=\\\'https://huggingface.co/\\\'+repo+\\\'/resolve/\\\'+revision+\\\'/\\\'+quote(item[\\\'rfilename\\\'],safe=\\\'/\\\')\\n        def verify_progress(value, name=name, size=size):\\n            now=time.monotonic()\\n            if now-last[0]>=.8 or value==size:\\n                emit_progress(\\\'verify\\\',\\\'正在校验下载的权重。\\\',value,size,name);last[0]=now\\n        download_file(http,url,folder/name,size,sha,progress,verify_progress)\\n        done += size\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps({\\\'base_repo\\\':BASE_REPO,\\\'base_revision\\\':base[\\\'sha\\\'],\\\'gguf_repo\\\':GGUF_REPO,\\\'gguf_revision\\\':quant[\\\'sha\\\'],\\\'download_bytes\\\':total},indent=2))\\n    emit_progress(\\\'download\\\',\\\'模型权重下载完成。\\\',done,total or None)\\n    return folder\\n\\n\\ndef copy_assets(source, target):\\n    source,target=Path(source),Path(target)\\n    files=[p for p in source.rglob(\\\'*\\\') if p.is_file()]\\n    total=sum(p.stat().st_size for p in files); done=0; last=0\\n    for path in files:\\n        destination=target/path.relative_to(source); destination.parent.mkdir(parents=True,exist_ok=True)\\n        with path.open(\\\'rb\\\') as src, destination.open(\\\'wb\\\') as dst:\\n            for chunk in iter(lambda:src.read(8*1024*1024),b\\\'\\\'):\\n                dst.write(chunk);done+=len(chunk)\\n                now=time.monotonic()\\n                if now-last>=.8:\\n                    emit_progress(\\\'copy\\\',\\\'正在准备已缓存的模型文件。\\\',done,total,path.name);last=now\\n    emit_progress(\\\'copy\\\',\\\'模型文件准备完成。\\\',done,total)\\n\\n"""Private, create-only Kaggle cache of prepared INT8/Q4 model assets.\\n\\nExecuted in the user\\\'s Kaggle kernel. Credentials are passed at connection time,\\nused only for Kaggle operations, and removed by the caller\\\'s finally block.\\n"""\\nimport contextlib\\nimport tempfile\\nimport threading\\nfrom concurrent.futures import ThreadPoolExecutor\\n\\nCACHE_VERSION = 1\\n\\n\\ndef cache_file_hashes(folder):\\n    hashes = {}\\n    for file in Path(folder).rglob(\\\'*\\\'):\\n        if file.is_file():\\n            with file.open(\\\'rb\\\') as content:\\n                hashes[file.relative_to(folder).as_posix()] = hashlib.file_digest(content,\\\'sha256\\\').hexdigest()\\n    return hashes\\n\\n\\n@contextlib.contextmanager\\ndef cache_credentials(handle, token):\\n    owner = handle.split(\\\'/\\\')[0]\\n    values = {\\\'KAGGLE_USERNAME\\\': owner,\\\'KAGGLE_API_TOKEN\\\':None,\\\'KAGGLE_KEY\\\':None}\\n    if token.startswith(\\\'KGAT_\\\'): values[\\\'KAGGLE_API_TOKEN\\\'] = token\\n    else: values[\\\'KAGGLE_KEY\\\'] = token\\n    old = {name: os.environ.get(name) for name in values}\\n    try:\\n        for name,value in values.items():\\n            if value is None: os.environ.pop(name,None)\\n            else: os.environ[name]=value\\n        yield\\n    finally:\\n        for name, value in old.items():\\n            if value is None: os.environ.pop(name, None)\\n            else: os.environ[name] = value\\n\\n\\ndef cache_metadata(handle, token):\\n    owner, slug = handle.split(\\\'/\\\')\\n    with requests.Session() as http:\\n        if token.startswith(\\\'KGAT_\\\'): http.headers[\\\'Authorization\\\'] = \\\'Bearer \\\' + token\\n        else: http.auth = (owner, token)\\n        response = http.get(\\\'https://www.kaggle.com/api/v1/datasets/view/\\\'+owner+\\\'/\\\'+slug, timeout=(20,60))\\n        if response.status_code in (403,404):\\n            # Kaggle hides missing private slugs behind 403. Verify absence in\\n            # the authenticated owner listing; a permission failure stays fatal.\\n            for page in range(1,101):\\n                listing = http.get(\\\'https://www.kaggle.com/api/v1/datasets/list\\\',\\n                    params={\\\'user\\\':owner,\\\'search\\\':slug,\\\'page\\\':page}, timeout=(20,60))\\n                listing.raise_for_status()\\n                records = listing.json()\\n                if not isinstance(records,list): raise RuntimeError(\\\'Invalid dataset listing\\\')\\n                if any(item.get(\\\'ref\\\') == handle for item in records):\\n                    raise RuntimeError(\\\'Existing cache is inaccessible\\\')\\n                if len(records)<20: return None\\n            raise RuntimeError(\\\'Cannot confirm cache absence\\\')\\n        response.raise_for_status()\\n        data = response.json()\\n        if data.get(\\\'isPrivate\\\') is not True:\\n            raise RuntimeError(\\\'模型缓存必须为私有 Dataset；没有修改现有 Dataset。\\\')\\n        return data\\n\\n\\ndef validate_cache_manifest(manifest):\\n    generic=manifest.get(\\\'studio_model_id\\\') is not None\\n    if generic:\\n        if (manifest.get(\\\'studio_cache_version\\\')!=2 or manifest[\\\'studio_model_id\\\']!=globals().get(\\\'_CC_MODEL_ID\\\')\\n            or manifest.get(\\\'model_sources\\\')!=globals().get(\\\'_CC_MODEL_SOURCES\\\')):\\n            raise RuntimeError(\\\'所选模型缓存版本不兼容。\\\')\\n    elif (manifest.get(\\\'studio_cache_version\\\') != CACHE_VERSION\\n        or manifest.get(\\\'base_revision\\\') != BASE_REVISION\\n        or manifest.get(\\\'gguf_revision\\\') != GGUF_REVISION):\\n        raise RuntimeError(\\\'同名 Dataset 不是兼容的本程序缓存，没有覆盖它。\\\')\\n    files = manifest.get(\\\'cache_files\\\', {})\\n    if not files or (not generic and \\\'transformer-q4.gguf\\\' not in files) or (generic and not any(name.endswith(\\\'.safetensors\\\') for name in files)):\\n        raise RuntimeError(\\\'模型缓存文件清单不完整。\\\')\\n    for name, size in files.items():\\n        safe_relative(name)\\n        if not isinstance(size,int) or size<0: raise RuntimeError(\\\'Invalid cached file size\\\')\\n    return manifest\\n\\n\\ndef validate_cache(folder):\\n    folder = Path(folder)\\n    manifest = validate_cache_manifest(json.loads((folder/\\\'asset-sources.json\\\').read_text()))\\n    for name, size in manifest[\\\'cache_files\\\'].items():\\n        file = folder/name\\n        if not file.is_file() or file.stat().st_size != size:\\n            raise RuntimeError(\\\'模型缓存文件不完整。\\\')\\n    return manifest\\n\\n\\ndef cache_http(handle, token):\\n    http = requests.Session()\\n    if token.startswith(\\\'KGAT_\\\'): http.headers[\\\'Authorization\\\'] = \\\'Bearer \\\'+token\\n    else: http.auth = (handle.split(\\\'/\\\')[0],token)\\n    return http\\n\\n\\ndef cache_file_url(handle, version, name):\\n    safe_relative(name)\\n    return (\\\'https://www.kaggle.com/api/v1/datasets/download/\\\'+handle+\\\'/\\\'+quote(name,safe=\\\'\\\')+\\n        \\\'?raw=true&dataset_version_number=\\\'+str(int(version)))\\n\\n\\ndef read_cache_manifest(http, handle, version):\\n    response = http.get(cache_file_url(handle,version,\\\'asset-sources.json\\\'),timeout=(20,60))\\n    response.raise_for_status()\\n    return validate_cache_manifest(response.json())\\n\\n\\ndef download_cache_assets(folder, handle, token, metadata):\\n    folder = Path(folder); folder.mkdir(parents=True,exist_ok=True)\\n    version = metadata[\\\'currentVersionNumber\\\']\\n    with cache_http(handle,token) as http:\\n        manifest = read_cache_manifest(http,handle,version)\\n    total = sum(manifest[\\\'cache_files\\\'].values()); received={}; last=[0.0]\\n    progress_lock=threading.Lock()\\n    def transfer(item):\\n        name,size=item\\n        def progress(value):\\n            with progress_lock:\\n                received[name]=value\\n                now=time.monotonic()\\n                if now-last[0]>=.8 or value==size:\\n                    emit_progress(\\\'cache-download\\\',\\\'正在拉取私有 Dataset 模型缓存。\\\',sum(received.values()),total,name)\\n                    last[0]=now\\n        with cache_http(handle,token) as http:\\n            download_file(http,cache_file_url(handle,version,name),folder/name,size,\\n                manifest.get(\\\'cache_sha256\\\',{}).get(name),progress)\\n    # Independent resumable streams avoid serializing all multi-GB shards.\\n    files=sorted(manifest[\\\'cache_files\\\'].items(),key=lambda item:item[1],reverse=True)\\n    with ThreadPoolExecutor(max_workers=4) as pool:\\n        list(pool.map(transfer,files))\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps(manifest,indent=2))\\n    validate_cache(folder)\\n    return folder\\n\\n\\ndef prepare_auto_assets(folder):\\n    handle, token = globals().get(\\\'_QWEN_CACHE_HANDLE\\\'), globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'disabled\\\'\\n    globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\'] = handle\\n    if handle and token:\\n        try:\\n            emit_progress(\\\'cache-check\\\', \\\'正在检查你的私有模型缓存。\\\')\\n            metadata = cache_metadata(handle, token)\\n            if metadata:\\n                emit_progress(\\\'cache-download\\\', \\\'正在拉取私有 Dataset 模型缓存。\\\')\\n                download_cache_assets(folder, handle, token, metadata)\\n                globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n                globals()[\\\'_QWEN_RESOLVED_SOURCE\\\'] = \\\'kaggle-cache\\\'\\n                return Path(folder), False\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'missing\\\'\\n        except Exception:\\n            # A metadata/auth/compatibility error must never permit overwriting\\n            # an existing dataset. Generation can still use the public weights.\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'unavailable\\\'\\n            emit_progress(\\\'cache-warning\\\', \\\'私有缓存暂不可用，本次从 Hugging Face 准备；没有修改已有 Dataset。\\\')\\n    globals()[\\\'_QWEN_RESOLVED_SOURCE\\\'] = \\\'huggingface\\\'\\n    return prepare_hf_assets(folder), True\\n\\n\\ndef save_private_cache(folder, encoder):\\n    handle, token = globals().get(\\\'_QWEN_CACHE_HANDLE\\\'), globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    status = globals().get(\\\'_QWEN_CACHE_STATUS\\\', \\\'missing\\\') if globals().get(\\\'_QWEN_CACHE_STATUS_HANDLE\\\') == handle else \\\'missing\\\'\\n    globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\'] = handle\\n    if not handle or not token or status in (\\\'saved\\\',\\\'unavailable\\\'): return\\n    try:\\n        emit_progress(\\\'cache-check\\\', \\\'正在确认私有缓存状态。\\\')\\n        metadata = cache_metadata(handle, token)\\n        if metadata:\\n            with cache_http(handle,token) as http:\\n                read_cache_manifest(http,handle,metadata[\\\'currentVersionNumber\\\'])\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n            return\\n        folder = Path(folder)\\n        emit_progress(\\\'cache-prepare\\\', \\\'正在保存 INT8 编码器和 Q4 权重，首次需要额外等待。\\\')\\n        with tempfile.TemporaryDirectory(prefix=\\\'qwen-private-cache-\\\', dir=\\\'/kaggle/temp\\\') as temporary:\\n            bundle = Path(temporary)\\n            generic=globals().get(\\\'_CC_MODEL_ID\\\',\\\'qwen-image-2.1\\\')!=\\\'qwen-image-2.1\\\'\\n            if generic:\\n                def link_or_copy(src,dst):\\n                    try:os.link(src,dst)\\n                    except OSError:shutil.copy2(src,dst)\\n                shutil.copytree(folder,bundle,dirs_exist_ok=True,copy_function=link_or_copy,\\n                    ignore=shutil.ignore_patterns(\\\'*.partial\\\',\\\'*.verified.json\\\',\\\'asset-sources.json\\\'))\\n            for name in (() if generic else (\\\'processor\\\',\\\'scheduler\\\',\\\'vae\\\',\\\'transformer\\\')):\\n                shutil.copytree(folder/name, bundle/name, ignore=shutil.ignore_patterns(\\\'*.partial\\\',\\\'*.verified.json\\\'))\\n            for name in (() if generic else (\\\'transformer-q4.gguf\\\',\\\'model_index.json\\\',\\\'LICENSE\\\')):\\n                if (folder/name).exists(): shutil.copy2(folder/name, bundle/name)\\n            if not generic:encoder.save_pretrained(bundle/\\\'text_encoder\\\', safe_serialization=True, max_shard_size=\\\'4GB\\\')\\n            manifest = json.loads((folder/\\\'asset-sources.json\\\').read_text())\\n            manifest.update(studio_cache_version=2 if generic else CACHE_VERSION,\\n                cache_files={p.relative_to(bundle).as_posix():p.stat().st_size for p in bundle.rglob(\\\'*\\\') if p.is_file()},\\n                cache_sha256=cache_file_hashes(bundle))\\n            (bundle/\\\'asset-sources.json\\\').write_text(json.dumps(manifest, indent=2))\\n            validate_cache(bundle)\\n            emit_progress(\\\'cache-upload\\\', \\\'正在上传私有 Dataset；Kaggle 将显示文件上传进度。\\\')\\n            # Use KaggleHub\\\'s create-only primitive: its public upload helper\\n            # creates a new version on conflict, which could overwrite user data.\\n            from kagglehub import datasets\\n            api = datasets.dataset_upload.__globals__\\n            create = datasets.create_dataset_or_version.__globals__[\\\'_create_dataset\\\']\\n            with cache_credentials(handle, token):\\n                uploads = api[\\\'upload_files_and_directories\\\'](str(bundle), item_type=api[\\\'ApiBlobType\\\'].DATASET,\\n                    ignore_patterns=api[\\\'normalize_patterns\\\'](default=api[\\\'DEFAULT_IGNORE_PATTERNS\\\'], additional=None))\\n                create(api[\\\'parse_dataset_handle\\\'](handle), uploads)\\n            # The create primitive explicitly sends is_private=True. Confirm\\n            # privacy again from the server before reporting success.\\n            emit_progress(\\\'cache-processing\\\', \\\'文件已上传，Kaggle 正在处理私有 Dataset。\\\')\\n            deadline = time.monotonic()+180\\n            while not cache_metadata(handle, token):\\n                if time.monotonic() >= deadline: raise RuntimeError(\\\'Dataset metadata not ready\\\')\\n                time.sleep(5)\\n        globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'saved\\\'\\n        emit_progress(\\\'cache-saved\\\', \\\'私有模型缓存已保存，下次新会话优先拉取，无需重新下载或量化。\\\')\\n    except Exception as error:\\n        globals()[\\\'_QWEN_CACHE_STATUS\\\'] = \\\'failed\\\'\\n        globals()[\\\'_QWEN_CACHE_FAILURE_TYPE\\\'] = type(error).__name__\\n        emit_progress(\\\'cache-warning\\\', \\\'私有缓存保存未完成；本次仍可正常生图，重新连接后会再次检查。\\\')\\n\\n"""Public, pinned, resumable model downloads. Executed after asset_manager.py."""\\n\\ndef model_asset_records(model_id, sources, http):\\n    records=[]\\n    for index,(repo,revision) in enumerate(sources):\\n        manifest=repository_manifest(repo,http,revision)\\n        if manifest.get(\\\'sha\\\')!=revision:raise RuntimeError(\\\'Model revision mismatch\\\')\\n        for item in manifest[\\\'siblings\\\']:\\n            name=item[\\\'rfilename\\\'];safe_relative(name)\\n            keep=False\\n            if model_id==\\\'z-image-turbo\\\':\\n                keep=(name in (\\\'model_index.json\\\',\\\'LICENSE\\\',\\\'LICENSE.txt\\\',\\\'README.md\\\') or\\n                      name.split(\\\'/\\\')[0] in (\\\'text_encoder\\\',\\\'tokenizer\\\',\\\'transformer\\\',\\\'vae\\\',\\\'scheduler\\\'))\\n                keep=keep and name.endswith((\\\'.json\\\',\\\'.safetensors\\\',\\\'.txt\\\',\\\'.model\\\',\\\'.jinja\\\',\\\'.tiktoken\\\',\\\'.md\\\'))\\n            elif index==0:\\n                # Lightning replaces the base UNet; do not download a second one.\\n                parts=name.split(\\\'/\\\')\\n                keep=name in (\\\'model_index.json\\\',\\\'LICENSE.md\\\',\\\'README.md\\\',\\\'unet/config.json\\\')\\n                if parts[0] in (\\\'text_encoder\\\',\\\'text_encoder_2\\\',\\\'tokenizer\\\',\\\'tokenizer_2\\\',\\\'scheduler\\\',\\\'vae\\\'):\\n                    keep=name.endswith((\\\'.json\\\',\\\'.txt\\\',\\\'.model\\\',\\\'.jinja\\\')) or name.endswith(\\\'.fp16.safetensors\\\')\\n            else:keep=name in (\\\'sdxl_lightning_4step_unet.safetensors\\\',\\\'LICENSE.md\\\',\\\'README.md\\\')\\n            if keep:records.append((repo,revision,item,(\\\'base/\\\' if index==0 else \\\'lightning/\\\')+name if model_id==\\\'sdxl-lightning\\\' else name))\\n    if not any(name.endswith(\\\'.safetensors\\\') for _,_,_,name in records):raise RuntimeError(\\\'Model weights missing\\\')\\n    return records\\n\\n\\ndef prepare_model_hf_assets(folder):\\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\\n    with requests.Session() as http:\\n        emit_progress(\\\'manifest\\\',\\\'正在获取所选模型清单。\\\')\\n        records=model_asset_records(_CC_MODEL_ID,_CC_MODEL_SOURCES,http)\\n        total=sum(int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0) for _,_,item,_ in records)\\n        done=0;last=[0.0]\\n        for repo,revision,item,name in records:\\n            size=int(item.get(\\\'size\\\') or item.get(\\\'lfs\\\',{}).get(\\\'size\\\') or 0)\\n            if size<=0:raise RuntimeError(\\\'Model file size missing\\\')\\n            start=done\\n            def progress(value,name=name,start=start,size=size):\\n                now=time.monotonic()\\n                if now-last[0]>.8 or value==size:\\n                    emit_progress(\\\'download\\\',\\\'正在下载所选模型。\\\',start+value,total,name);last[0]=now\\n            url=\\\'https://huggingface.co/\\\'+repo+\\\'/resolve/\\\'+revision+\\\'/\\\'+quote(item[\\\'rfilename\\\'],safe=\\\'/\\\')\\n            download_file(http,url,folder/name,size,item.get(\\\'lfs\\\',{}).get(\\\'sha256\\\'),progress)\\n            done+=size\\n    manifest={\\\'studio_model_id\\\':_CC_MODEL_ID,\\\'model_sources\\\':_CC_MODEL_SOURCES,\\\'download_bytes\\\':total}\\n    (folder/\\\'asset-sources.json\\\').write_text(json.dumps(manifest,indent=2))\\n    emit_progress(\\\'download\\\',\\\'所选模型下载完成。\\\',total,total)\\n    return folder\\n\\n\\ndef prepare_selected_model_assets(folder):\\n    handle,token=globals().get(\\\'_QWEN_CACHE_HANDLE\\\'),globals().get(\\\'_QWEN_CACHE_TOKEN\\\')\\n    globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'disabled\\\';globals()[\\\'_QWEN_CACHE_STATUS_HANDLE\\\']=handle\\n    if handle and token:\\n        try:\\n            emit_progress(\\\'cache-check\\\',\\\'正在检查所选模型的私有缓存。\\\')\\n            metadata=cache_metadata(handle,token)\\n            if metadata:\\n                download_cache_assets(folder,handle,token,metadata)\\n                globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'saved\\\';globals()[\\\'_QWEN_RESOLVED_SOURCE\\\']=\\\'kaggle-cache\\\'\\n                return Path(folder)\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'missing\\\'\\n        except Exception:\\n            globals()[\\\'_QWEN_CACHE_STATUS\\\']=\\\'unavailable\\\'\\n            emit_progress(\\\'cache-warning\\\',\\\'正在从 Hugging Face 准备所选模型。\\\')\\n    globals()[\\\'_QWEN_RESOLVED_SOURCE\\\']=\\\'huggingface\\\'\\n    return prepare_model_hf_assets(folder)\\n\',\'<cc-model-assets>\',\'exec\'),globals())\n_existing_state=globals().get(\'_COMFY_QWEN_STATE\')\nif not (_existing_state and globals().get(\'pipe\') is not None and callable(globals().get(\'generate_image\'))):\n    if False:\n        import subprocess,sys\n        subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'bitsandbytes==0.50.2\',\'safetensors\',\'diffusers>=0.36,<0.43\',\'transformers>=4.51,<6\',\'accelerate>=1.2\'],check=True,timeout=900)\n    for _cc_partial in (\'pipe\',\'encoder\',\'transformer\',\'vae\',\'processor\',\'scheduler\',\'generate_image\'):\n        globals().pop(_cc_partial,None)\n    import gc\n    gc.collect()\n    import torch\n    torch.cuda.empty_cache()\n    exec(compile(\'"""Additional model loaders. The caller embeds pinned catalog and asset helpers."""\\nimport gc\\nimport time\\nimport torch\\n\\nassert torch.cuda.device_count()>=2,\\\'Select GPU T4 x2.\\\'\\nASSET_DIR=Path(\\\'/kaggle/temp/cc-media-models\\\')/_CC_MODEL_ID\\nstarted=time.perf_counter()\\nprepare_selected_model_assets(ASSET_DIR)\\n_assets_ready=time.perf_counter()\\nemit_progress(\\\'loading\\\',\\\'正在加载所选模型。\\\')\\nif _CC_MODEL_ID==\\\'z-image-turbo\\\':\\n    from diffusers import ZImagePipeline,ZImageTransformer2DModel,AutoencoderKL\\n    from transformers import Qwen3Model,BitsAndBytesConfig\\n    encoder=Qwen3Model.from_pretrained(ASSET_DIR/\\\'text_encoder\\\',torch_dtype=torch.float16,\\n        quantization_config=BitsAndBytesConfig(load_in_8bit=True),device_map={\\\'\\\':1},local_files_only=True)\\n    transformer=ZImageTransformer2DModel.from_pretrained(ASSET_DIR/\\\'transformer\\\',torch_dtype=torch.float16,local_files_only=True).to(\\\'cuda:0\\\')\\n    vae=AutoencoderKL.from_pretrained(ASSET_DIR/\\\'vae\\\',torch_dtype=torch.float32,local_files_only=True).to(\\\'cuda:0\\\')\\n    pipe=ZImagePipeline.from_pretrained(ASSET_DIR,torch_dtype=torch.float16,text_encoder=encoder,\\n        transformer=transformer,vae=vae,local_files_only=True)\\nelse:\\n    from diffusers import StableDiffusionXLPipeline,UNet2DConditionModel,EulerDiscreteScheduler\\n    from safetensors.torch import load_file\\n    transformer=UNet2DConditionModel.from_config(ASSET_DIR/\\\'base\\\',subfolder=\\\'unet\\\')\\n    transformer.load_state_dict(load_file(ASSET_DIR/\\\'lightning/sdxl_lightning_4step_unet.safetensors\\\',device=\\\'cpu\\\'))\\n    transformer=transformer.to(device=\\\'cuda:0\\\',dtype=torch.float16)\\n    pipe=StableDiffusionXLPipeline.from_pretrained(ASSET_DIR/\\\'base\\\',unet=transformer,\\n        torch_dtype=torch.float16,variant=\\\'fp16\\\',local_files_only=True).to(\\\'cuda:0\\\')\\n    pipe.scheduler=EulerDiscreteScheduler.from_config(pipe.scheduler.config,timestep_spacing=\\\'trailing\\\')\\n    encoder=pipe.text_encoder\\nload_stats={\\\'copy_seconds\\\':round(_assets_ready-started,2),\\\'loading_seconds\\\':round(time.perf_counter()-_assets_ready,2)}\\n\\n\\n@torch.inference_mode()\\ndef generate_image(prompt,seed=42,width=512,height=512,steps=None):\\n    steps=_CC_MODEL_PROFILE[\\\'default_steps\\\'] if steps is None else steps\\n    if _CC_MODEL_ID==\\\'sdxl-lightning\\\' and steps!=4:raise ValueError(\\\'SDXL-Lightning requires 4 steps\\\')\\n    for i in range(2):torch.cuda.reset_peak_memory_stats(i)\\n    started=time.perf_counter();encoding_seconds=0\\n    if _CC_MODEL_ID==\\\'z-image-turbo\\\':\\n        positive,_=pipe.encode_prompt(prompt,device=torch.device(\\\'cuda:1\\\'),do_classifier_free_guidance=False)\\n        torch.cuda.synchronize(1);encoding_seconds=time.perf_counter()-started\\n        kwargs={\\\'prompt_embeds\\\':[value.to(\\\'cuda:0\\\') for value in positive],\\\'guidance_scale\\\':0.0}\\n    else:\\n        positive,negative,pooled,negative_pooled=pipe.encode_prompt(prompt=prompt,device=torch.device(\\\'cuda:0\\\'),do_classifier_free_guidance=False)\\n        torch.cuda.synchronize(0);encoding_seconds=time.perf_counter()-started\\n        kwargs={\\\'prompt_embeds\\\':positive,\\\'pooled_prompt_embeds\\\':pooled,\\\'guidance_scale\\\':0.0}\\n    # Explicit denoising/decode timing; CUDA events avoid synchronizing every step.\\n    timings=[]\\n    def record_step(pipeline,index,timestep,callback_kwargs):\\n        event=torch.cuda.Event(enable_timing=True);event.record(torch.cuda.current_stream(0));timings.append(event)\\n        return callback_kwargs\\n    before=torch.cuda.Event(enable_timing=True);before.record(torch.cuda.current_stream(0))\\n    result=pipe(width=width,height=height,num_inference_steps=steps,\\n        generator=torch.Generator(device=\\\'cuda:0\\\').manual_seed(seed),callback_on_step_end=record_step,**kwargs).images[0]\\n    after=torch.cuda.Event(enable_timing=True);after.record(torch.cuda.current_stream(0));torch.cuda.synchronize(0)\\n    denoise=before.elapsed_time(timings[-1])/1000 if timings else None\\n    decode=timings[-1].elapsed_time(after)/1000 if timings else None\\n    return result,{\\\'model_id\\\':_CC_MODEL_ID,\\\'encoding_seconds\\\':round(encoding_seconds,3),\\n        \\\'generation_seconds\\\':round(time.perf_counter()-started-encoding_seconds,3),\\n        \\\'denoising_seconds\\\':round(denoise,3) if denoise is not None else None,\\n        \\\'decode_seconds\\\':round(decode,3) if decode is not None else None,\\n        \\\'denoising_steps\\\':len(timings),\\\'milliseconds_per_step\\\':round(denoise*1000/len(timings),2) if timings else None,\\n        \\\'attention_backend\\\':\\\'sdpa\\\',\\\'parallel_strategy\\\':\\\'encoder_split\\\' if _CC_MODEL_ID==\\\'z-image-turbo\\\' else \\\'single_gpu\\\',\\n        \\\'width\\\':width,\\\'height\\\':height,\\\'steps\\\':steps,\\\'seed\\\':seed,\\n        \\\'peak_gpu_gib\\\':[round(torch.cuda.max_memory_allocated(i)/2**30,2) for i in range(2)]}\\n\',\'<cc-model-runtime>\',\'exec\'),globals())\n    _existing_state={\'dataset\':\'model/\'+_CC_MODEL_ID,\'model_id\':_CC_MODEL_ID,\'runtime_id\':uuid.uuid4().hex,\'load_count\':1,\'generations\':0,\'loaded_this_request\':True}\nelse:\n    _existing_state[\'loaded_this_request\']=False\n_COMFY_QWEN_STATE=_existing_state\nif globals().get(\'_QWEN_CACHE_HANDLE\') and globals().get(\'_QWEN_CACHE_TOKEN\'):\n    save_private_cache(ASSET_DIR,encoder)\nprint(\'COMFY_SESSION_READY:\'+json.dumps({\'runtime_id\':_existing_state[\'runtime_id\'],\'load_count\':_existing_state[\'load_count\'],\'reused\':not _existing_state[\'loaded_this_request\']}),flush=True)\n'}
def generation_code(prompt, seed, steps, model_id=DEFAULT_MODEL):
    model_profile(model_id)
    args = repr(json.dumps({'prompt': prompt, 'seed': seed, 'steps': steps}, ensure_ascii=False))
    return (
        'import json, io, base64\n' + \
        'assert globals().get("_CC_MODEL_ID","qwen-image-2.1") == '+repr(model_id)+', "Active model changed; reconnect before generating."\n' + \
        'args = json.loads(' + args + ')\n'
        "im, stats = generate_image(args['prompt'], seed=args['seed'], steps=args['steps'], width=512, height=512)\n"
        "_COMFY_QWEN_STATE['generations'] += 1\n"
        "stats = dict(stats, model_id=globals().get('_CC_MODEL_ID','qwen-image-2.1'), runtime_id=_COMFY_QWEN_STATE['runtime_id'], "
        "model_load_count=_COMFY_QWEN_STATE['load_count'], "
        "model_reused=not _COMFY_QWEN_STATE['loaded_this_request'], "
        "generation_number=_COMFY_QWEN_STATE['generations'], asset_source=globals().get('_QWEN_RESOLVED_SOURCE',globals().get('_QWEN_ASSET_MODE','dataset')), "
        "cache_status=globals().get('_QWEN_CACHE_STATUS','disabled'))\n"
        "_COMFY_QWEN_STATE['loaded_this_request'] = False\n"
        "buf = io.BytesIO()\nim.save(buf, format='PNG')\n"
        "print('COMFY_RESULT:' + json.dumps({'png': base64.b64encode(buf.getvalue()).decode(), 'stats': stats}), flush=True)\n"
    )

model = widgets.Dropdown(options=[(p['name'],key) for key,p in MODELS.items()],value=DEFAULT_MODEL,description='Model / 模型',layout=widgets.Layout(width='95%'))
prompt = widgets.Textarea(value='A small orange fox beside a ceramic teacup, soft morning window light, natural photography', description='Prompt / 提示词', layout=widgets.Layout(width='95%', height='110px'))
steps = widgets.IntSlider(value=20,min=1,max=50,description='Steps / 步数')
def model_changed(change):
    profile=model_profile(model.value);steps.value=profile['default_steps'];steps.disabled=profile['fixed_steps'] is not None
model.observe(model_changed,names='value')
seed = widgets.IntText(value=42,description='Seed / 种子')
button = widgets.Button(description='Generate / 生成',button_style='primary',icon='magic')
progress = widgets.IntProgress(value=0,min=0,max=100,description='Setup / 准备',bar_style='info',layout=widgets.Layout(width='95%'))
status = widgets.HTML('<span style="color:#777">第一次将从 Hugging Face 自动下载权重，无需 Dataset 或额外 Token。</span>')
image_view = widgets.Image(format='png',layout=widgets.Layout(max_width='512px'))
info = widgets.HTML()
cache_user = widgets.Text(description='Kaggle user / 账号',placeholder='可选：填写后启用私有 Dataset 缓存')
cache_token = widgets.Password(description='API Key',placeholder='可选：仅当前内核使用')
cache_box = widgets.Accordion(children=[widgets.VBox([cache_user,cache_token])])
cache_box.set_title(0,'Private cache / 私有缓存（可选 optional）')
cache_box.selected_index=None

def on_progress(event):
    message=html.escape(event.get('message',''))
    if event.get('total_bytes'):
        percent=event.get('percent',0)
        progress.value=int(percent)
        message += ' · %.1f%% · %.2f / %.2f GB' % (percent,event['downloaded_bytes']/1e9,event['total_bytes']/1e9)
    else:progress.value=0
    status.value='<span style="color:#777">'+message+'</span>'

def run(_):
    if not prompt.value.strip():
        status.value='请填写提示词。 / Enter a prompt.'
        return
    button.disabled=True;model.disabled=True
    started=time.perf_counter()
    try:
        globals()['_QWEN_PROGRESS_CALLBACK']=on_progress
        capture=io.StringIO()
        if cache_user.value.strip() and cache_token.value.strip():
            import re
            if not re.fullmatch(r'[A-Za-z0-9_-]+',cache_user.value.strip()):raise ValueError('Kaggle 用户名格式不正确')
            globals()['_QWEN_CACHE_HANDLE']=cache_user.value.strip()+'/'+model_profile(model.value)['cache_slug']
            globals()['_QWEN_CACHE_TOKEN']=cache_token.value.strip()
        try:
            with contextlib.redirect_stdout(capture):exec(_BOOTSTRAPS[model.value],globals())
        finally:
            globals().pop('_QWEN_CACHE_TOKEN',None)
            globals().pop('_QWEN_CACHE_HANDLE',None)
            cache_token.value=''
        status.value='模型已就绪，正在生成… / Model ready, generating…'
        progress.value=0
        capture=io.StringIO()
        with contextlib.redirect_stdout(capture):exec(generation_code(prompt.value,seed.value,steps.value,model.value),globals())
        line=next(line for line in capture.getvalue().splitlines() if line.startswith('COMFY_RESULT:'))
        result=json.loads(line[len('COMFY_RESULT:'):])
        raw=base64.b64decode(result['png'])
        Image.open(io.BytesIO(raw)).verify()
        image_view.value=raw
        progress.value=100
        stats=result['stats']
        status.value='生成完成 · '+('已复用模型' if stats['model_reused'] else '首次准备完成，后续复用')
        info.value='本次 %.1f 秒 · 模型加载 %d 次 · PNG 512 × 512' % (time.perf_counter()-started,stats['model_load_count'])
        button.description='Generate / 生成'
    except Exception as error:
        status.value='<span style="color:#b94747">'+html.escape(str(error))+'</span>'
    finally:button.disabled=False;model.disabled=False

button.on_click(run)
display(widgets.VBox([widgets.HTML('<h2 style="color:#6258ee">CC Media Studio · Kaggle</h2>'),model,prompt,steps,seed,cache_box,button,progress,status,image_view,info]))
